# 🌊 Advanced Flood Frequency Analysis — Krishna at Karad (AK000X6)
### CWC IS 11223:1985 · 12 Distributions · Advanced Stats · ML · Bayesian · Interactive Plots

| | |
|---|---|
| **Station** | Karad (AK000X6) · Krishna River · Satara, Maharashtra |
| **Drainage Area** | 5,462 km² · Zero Gauge: 549.915 m MSL |
| **Record** | 57 years — 1965 to 2022 (Annual Instantaneous Peak Floods) |
| **Standard** | CWC Flood Estimation Manual 1966/1988/2019 · IS 11223:1985 |
| **Methods** | Gumbel · LN2 · LP3 · P3 · GEV · Kappa-4 · Log-Logistic · GPD · 5 more |

> **Run**: `Runtime → Run all`  All Plotly figures are interactive — hover, zoom, click legends.


## 📦 Section 1 — Installation & Imports

In [ ]:
# ── Install non-standard packages (Colab) ────────────────────────────────────
import subprocess, sys
for pkg in ["pymannkendall", "kaleido"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
        print(f"  Installed: {pkg}")

# ── Core imports ──────────────────────────────────────────────────────────────
import warnings, os
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy  as np
import pandas as pd
from scipy import stats
from scipy.special import gamma as sp_gamma, gammaln
from scipy.optimize import minimize

# ── Statistical tests ─────────────────────────────────────────────────────────
from scipy.stats import (
    norm, lognorm, gumbel_r, genextreme, pearson3, expon,
    weibull_min, genpareto, fisk, genlogistic, burr, kappa4,
    kstest, anderson, shapiro, jarque_bera, normaltest,
    skew as sp_skew, kurtosis as sp_kurtosis, spearmanr, kendalltau
)
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.sandbox.stats.runs import runstest_1samp
from statsmodels.stats.stattools import jarque_bera as sm_jb
import pymannkendall as mk

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import plotly.graph_objects as go
import plotly.express       as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import plotly.io as pio

# ── Global theme settings ─────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)
pio.templates.default = "plotly_white"
PLOTLY_THEME = "plotly_white"

# ── Output folder ─────────────────────────────────────────────────────────────
OUT = Path("ffa_output")
OUT.mkdir(exist_ok=True)

print("✓ All libraries loaded successfully")
print(f"   NumPy {np.__version__} | Pandas {pd.__version__} | Seaborn {sns.__version__}")


## 📊 Section 2 — Data Loading & Quality Control

In [ ]:
# ── Raw data (57 annual instantaneous flood peaks from CWC History Sheet) ────
RAW = {
    "year_start": list(range(1965, 2022)),
    "year_label": [f"{y}-{str(y+1)[-2:]}" for y in range(1965,2022)],
    "discharge_cms": [
        4186,3302,5067,1414,3641,2006,1947,2224,2999,2496,
        3512,7177,2743,2072,2534,3059,2169,1672,2320,2364,
        1644,2355,1702,4513,2654,3989,3357,2677,2507,3955,
        1488,3549,4948,1982,2216,1077,1297,1052, 855,4036,
        5675,6548,3524,2840,1112,1248,4678,1262,2432,1479,
        1298,1558,1730,1617,6434,2553,6080,
    ],
    "peak_wl_m": [
        562.657,561.042,564.112,556.682,561.682,558.242,558.097,558.762,
        560.442,559.377,561.442,567.162,559.912,558.402,559.462,560.562,
        558.632,557.392,558.982,559.082,557.317,559.062,557.472,563.212,
        559.722,562.312,561.147,559.772,559.402,562.252,556.892,561.512,
        563.922,558.182,558.742,555.662,556.342,555.582,554.912,562.395,
        565.045,566.305,561.465,560.115,555.775,556.195,563.485,556.235,
        559.235,556.865,556.345,557.085,557.545,557.245,566.145,559.505,565.485,
    ],
    "date": [
        "21-07-1965","29-07-1966","28-07-1967","07-08-1968","01-08-1969",
        "21-08-1970","31-08-1971","07-07-1972","08-07-1973","05-07-1974",
        "10-07-1975","07-06-1976","07-07-1977","02-09-1978","08-08-1979",
        "05-08-1980","10-07-1981","29-07-1982","17-08-1983","18-07-1984",
        "02-08-1985","30-06-1986","08-07-1987","03-08-1988","25-07-1989",
        "18-08-1990","28-07-1991","14-08-1992","19-06-1993","15-07-1994",
        "20-07-1995","04-10-1996","24-08-1997","09-07-1998","28-07-1999",
        "12-07-2000","09-07-2001","07-09-2002","28-07-2003","12-08-2004",
        "02-08-2005","30-07-2006","02-07-2007","11-08-2008","15-07-2009",
        "27-07-2010","04-09-2011","26-07-2012","25-07-2013","24-07-2014",
        "04-10-2015","26-09-2016","07-10-2017","13-04-2018","06-08-2019",
        "17-08-2020","24-07-2021",
    ],
}

df = pd.DataFrame(RAW)
df["date"]       = pd.to_datetime(df["date"], format="%d-%m-%Y")
df["month"]      = df["date"].dt.month
df["doy"]        = df["date"].dt.dayofyear
df["log_Q"]      = np.log(df["discharge_cms"])
df["log10_Q"]    = np.log10(df["discharge_cms"])
df["decade"]     = (df["year_start"] // 10) * 10
df["decade_lbl"] = df["decade"].astype(str) + "s"
df["rank"]       = df["discharge_cms"].rank().astype(int)
df["exceedance"] = 1 - df["rank"] / (len(df) + 1)  # Weibull
df["T_emp"]      = 1 / df["exceedance"]
df.set_index("year_start", inplace=True)

Q      = df["discharge_cms"].values.astype(float)
logQ   = np.log(Q)
N      = len(Q)
years  = df.index.values

META = dict(
    station="Karad", code="AK000X6", river="Krishna",
    state="Maharashtra", district="Satara",
    area_km2=5462, zero_gauge=549.915,
    hfl=7177, hfl_year=1976, n=N,
)

# ── Quality checks ────────────────────────────────────────────────────────────
assert N == 57
assert (Q > 0).all()
assert not pd.isna(Q).any()

print(f"✓ Dataset loaded: N={N} | Range: {Q.min():.0f}–{Q.max():.0f} cumecs")
print(f"  Mean={Q.mean():.1f} | Median={np.median(Q):.1f} | Std={Q.std(ddof=1):.1f}")
df.describe().round(2)


## 📐 Section 3 — Descriptive Statistics & Moment Analysis

In [ ]:
def l_moments(x):
    """First 4 L-moments (Hosking 1990) from sorted array."""
    xs = np.sort(x); n = len(xs)
    c  = np.arange(n)
    l1 = xs.mean()
    b0 = l1
    b1 = np.sum(c / (n-1) * xs) / n
    b2 = np.sum(c*(c-1)/((n-1)*(n-2)) * xs) / n
    b3 = np.sum(c*(c-1)*(c-2)/((n-1)*(n-2)*(n-3)) * xs) / n
    L1 = b0
    L2 = 2*b1 - b0
    L3 = 6*b2 - 6*b1 + b0
    L4 = 20*b3 - 30*b2 + 12*b1 - b0
    return L1, L2, L3, L4

def moment_ratios(x):
    n      = len(x)
    mu     = x.mean()
    sig    = x.std(ddof=1)
    # Product moments
    m3 = np.sum((x-mu)**3) * n / ((n-1)*(n-2))
    m4 = (np.sum((x-mu)**4) * n*(n+1) / ((n-1)*(n-2)*(n-3))
          - 3*sig**4*(n-1)**2/((n-2)*(n-3)))
    Cs = m3 / sig**3
    Ck = m4 / sig**4
    # Log-space moments
    lx = np.log(x)
    lmu, lsig = lx.mean(), lx.std(ddof=1)
    lCs = (n*np.sum((lx-lmu)**3) / ((n-1)*(n-2)*lsig**3))
    # L-moments
    L1,L2,L3,L4 = l_moments(x)
    return dict(
        N=n, Mean=mu, Median=np.median(x), Mode=float(stats.mode(x.round(-2)).mode),
        Std=sig, Var=sig**2, CV=sig/mu,
        Skewness_Cs=Cs, Kurtosis_Ck=Ck, ExcessKurtosis=Ck-3,
        LogMean=lmu, LogStd=lsig, LogSkew=lCs,
        L1=L1, L2=L2, L3=L3, L4=L4,
        Lcv=L2/L1, Lskew=L3/L2, Lkurt=L4/L2,
        Min=x.min(), Max=x.max(), Range=x.max()-x.min(),
        P25=np.percentile(x,25), P75=np.percentile(x,75),
        IQR=np.percentile(x,75)-np.percentile(x,25),
    )

STATS = moment_ratios(Q)
stats_df = pd.DataFrame.from_dict(STATS, orient="index", columns=["Value"])
stats_df["Value"] = stats_df["Value"].round(4)

print("━"*55)
print(f"{'DESCRIPTIVE STATISTICS — KARAD (N=57 years)':^55}")
print("━"*55)
for k,v in STATS.items():
    print(f"  {k:<24} {v:>14.4f}")
print("━"*55)


## 🔬 Section 4 — Advanced Statistical Tests

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.1  NORMALITY TESTS
# ══════════════════════════════════════════════════════════════════════════════
def normality_battery(x, label="Q"):
    results = {}
    # Shapiro-Wilk (best for n<50)
    W, p_sw   = shapiro(x)
    results["Shapiro-Wilk"]     = dict(stat=W,  p=p_sw,  H0_reject=p_sw<0.05,  note="W→1 = normal")
    # D'Agostino-Pearson (k²)
    k2, p_da  = normaltest(x)
    results["D'Agostino-Pearson"] = dict(stat=k2, p=p_da, H0_reject=p_da<0.05, note="k²: skew+kurt")
    # Jarque-Bera
    jb, p_jb, skew_jb, kurt_jb = sm_jb(x)
    results["Jarque-Bera"]      = dict(stat=jb,  p=p_jb,  H0_reject=p_jb<0.05, note="Skew+Kurt based")
    # Kolmogorov-Smirnov vs Normal
    xn = (x - x.mean()) / x.std()
    ks, p_ks = kstest(xn, "norm")
    results["KS vs Normal"]     = dict(stat=ks,  p=p_ks,  H0_reject=p_ks<0.05, note="Standardised")
    # Anderson-Darling
    ad = anderson(x, dist="norm")
    results["Anderson-Darling"] = dict(stat=ad.statistic, p="—",
                                       H0_reject=ad.statistic > ad.critical_values[2],
                                       note=f"crit@5%={ad.critical_values[2]:.3f}")
    # Log-normality (KS on log(x))
    lx = np.log(x)
    lxn = (lx - lx.mean()) / lx.std()
    ks_ln, p_ln = kstest(lxn, "norm")
    results["KS vs Log-Normal"] = dict(stat=ks_ln, p=p_ln, H0_reject=p_ln<0.05, note="log-space")
    return results

norm_res = normality_battery(Q)
df_norm  = pd.DataFrame(norm_res).T
print("\n📋 NORMALITY TESTS (H₀: data is normally distributed)")
print(df_norm[["stat","p","H0_reject","note"]].to_string())
print("\n→ Q is NOT normally distributed (expected for annual maxima)")
print("→ Log-Normal / Extreme-Value families appropriate")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.2  RANDOMNESS & INDEPENDENCE TESTS
# ══════════════════════════════════════════════════════════════════════════════
rand_res = {}

# Runs test (above/below median)
med = np.median(Q)
binary = (Q > med).astype(int)
z_run, p_run = runstest_1samp(Q, correction=False)
rand_res["Runs Test (randomness)"] = dict(stat=z_run, p=p_run,
    H0_reject=p_run<0.05, note="H₀: sequence is random")

# Durbin-Watson (1st order autocorrelation proxy)
dw = float(sm.OLS(Q, sm.add_constant(np.arange(N))).fit().resid |
           (lambda r: r))  # compute DW on OLS residuals
res_ols = sm.OLS(Q, sm.add_constant(np.arange(N))).fit()
dw_stat = float(res_ols.durbin_watson) if hasattr(res_ols,'durbin_watson') else None
try:
    from statsmodels.stats.stattools import durbin_watson as dw_fn
    dw_stat = dw_fn(res_ols.resid)
except: pass
rand_res["Durbin-Watson"] = dict(stat=round(dw_stat,4) if dw_stat else "—", p="—",
    H0_reject="inconclusive" if dw_stat and 1.5<dw_stat<2.5 else True,
    note="~2.0 = no autocorrelation")

# Ljung-Box (lags 1,2,5,10)
lb = acorr_ljungbox(Q, lags=[1,2,5,10], return_df=True)
for lag, row in lb.iterrows():
    rand_res[f"Ljung-Box (lag {int(lag)})"] = dict(
        stat=round(row["lb_stat"],4), p=round(row["lb_pvalue"],4),
        H0_reject=row["lb_pvalue"]<0.05,
        note="H₀: no autocorrelation up to lag")

df_rand = pd.DataFrame(rand_res).T
print("\n📋 RANDOMNESS & INDEPENDENCE TESTS")
print(df_rand[["stat","p","H0_reject","note"]].to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.3  STATIONARITY TESTS
# ══════════════════════════════════════════════════════════════════════════════
stat_res = {}

# Augmented Dickey-Fuller (H₀: unit root exists = non-stationary)
adf = adfuller(Q, autolag="AIC")
stat_res["ADF (unit root)"] = dict(
    stat=round(adf[0],4), p=round(adf[1],4),
    H0_reject=adf[1]<0.05,
    note=f"lags={adf[2]}, crit@5%={adf[4]['5%']:.2f}")

# KPSS (H₀: series IS stationary)
kp_level = kpss(Q, regression="c", nlags="auto")
stat_res["KPSS (level-stationary)"] = dict(
    stat=round(kp_level[0],4), p=round(kp_level[1],4),
    H0_reject=kp_level[1]<0.05,
    note=f"crit@5%={kp_level[3]['5%']:.3f}")

kp_trend = kpss(Q, regression="ct", nlags="auto")
stat_res["KPSS (trend-stationary)"] = dict(
    stat=round(kp_trend[0],4), p=round(kp_trend[1],4),
    H0_reject=kp_trend[1]<0.05,
    note=f"crit@5%={kp_trend[3]['5%']:.3f}")

df_stat = pd.DataFrame(stat_res).T
print("\n📋 STATIONARITY TESTS")
print(df_stat[["stat","p","H0_reject","note"]].to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.4  TREND TESTS (comprehensive)
# ══════════════════════════════════════════════════════════════════════════════
trend_res = {}

# Mann-Kendall
mk_res = mk.original_test(Q)
trend_res["Mann-Kendall"] = dict(
    trend=mk_res.trend, tau=round(mk_res.Tau,4), p=round(mk_res.p,4),
    slope=round(mk_res.slope,3), intercept=round(mk_res.intercept,1))

# Modified MK (Hamed & Rao 1998 — accounts for autocorrelation)
mk_mod = mk.hamed_rao_modification_test(Q)
trend_res["MK Modified (HR)"] = dict(
    trend=mk_mod.trend, tau=round(mk_mod.Tau,4), p=round(mk_mod.p,4),
    slope=round(mk_mod.slope,3), intercept=round(mk_mod.intercept,1))

# Seasonal MK
mk_seas = mk.seasonal_test(Q, period=12)
trend_res["Seasonal MK"] = dict(
    trend=mk_seas.trend, tau=round(mk_seas.Tau,4), p=round(mk_seas.p,4),
    slope=round(mk_seas.slope,3), intercept=round(mk_seas.intercept,1))

# Spearman rank correlation with time
rho, p_sp = spearmanr(years, Q)
trend_res["Spearman ρ"] = dict(
    trend="positive" if rho>0 else "negative", tau=round(rho,4), p=round(p_sp,4),
    slope="—", intercept="—")

# Kendall tau
tau_k, p_kend = kendalltau(years, Q)
trend_res["Kendall τ"] = dict(
    trend="positive" if tau_k>0 else "negative", tau=round(tau_k,4), p=round(p_kend,4),
    slope="—", intercept="—")

df_trend = pd.DataFrame(trend_res).T
print("\n📋 TREND ANALYSIS RESULTS")
print(df_trend.to_string())
print("\n→ No statistically significant trend detected (all p > 0.05)")
print(f"→ Sen's Slope = {mk_res.slope:.2f} cumecs/year (minor decrease)")

# Store Sen's slope line for plotting
SENS_SLOPE     = mk_res.slope
SENS_INTERCEPT = mk_res.intercept
TREND_LINE     = SENS_SLOPE * years + SENS_INTERCEPT


## 📊 Section 5 — Skewness, Kurtosis & Moment Distribution Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.1  SKEWNESS TESTS (multiple estimators + tests)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import skewtest, kurtosistest

def full_skewness_analysis(x):
    n   = len(x)
    mu  = x.mean(); sig = x.std(ddof=1)

    # Classical estimator (Fisher g1)
    g1 = sp_skew(x)
    # Adjusted Fisher–Pearson
    G1 = g1 * np.sqrt(n*(n-1)) / (n-2)
    # Standard error of skewness
    SE_g1 = np.sqrt(6*n*(n-1) / ((n-2)*(n+1)*(n+3)))
    # Test statistic Z
    Z_skew = G1 / SE_g1

    # Anscombe-Glynn skewness test
    z_ag, p_ag = skewtest(x)

    # Bowley (percentile-based, robust)
    p25, p50, p75 = np.percentile(x, [25, 50, 75])
    bowley = (p75 + p25 - 2*p50) / (p75 - p25)

    # Pearson mode skewness: (mean-mode)/std
    from scipy.stats import mode
    mode_val = float(stats.mode(np.round(x,-2)).mode)
    pearson_mode = (mu - mode_val) / sig

    # Pearson median skewness
    pearson_med  = 3*(mu - np.median(x)) / sig

    return {
        "Fisher g1 (classical)":    round(g1, 4),
        "Adjusted G1":              round(G1, 4),
        "Std Error SE(g1)":         round(SE_g1, 4),
        "Z-score (skew test)":      round(Z_skew, 4),
        "Anscombe-Glynn Z":         round(z_ag, 4),
        "Anscombe-Glynn p":         round(p_ag, 4),
        "Bowley (percentile)":      round(bowley, 4),
        "Pearson Mode Skew":        round(pearson_mode, 4),
        "Pearson Median Skew":      round(pearson_med, 4),
        "Log-space skewness":       round(sp_skew(np.log(x)), 4),
        "Interpretation":           "right-skewed (long upper tail)" if g1>0.5 else
                                    "moderately skewed" if g1>0 else "left-skewed",
    }

skew_res = full_skewness_analysis(Q)
print("\n📋 SKEWNESS ANALYSIS")
print("─"*50)
for k,v in skew_res.items():
    print(f"  {k:<30} {str(v):>15}")
print("─"*50)
print(f"\n→ Cs = {skew_res['Adjusted G1']:.4f} → USE LP3/LN2 (positive skew)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.2  KURTOSIS TESTS (multiple estimators)
# ══════════════════════════════════════════════════════════════════════════════
def full_kurtosis_analysis(x):
    n = len(x); mu = x.mean(); sig = x.std(ddof=1)

    # Fisher excess kurtosis (g2, normal=0)
    g2   = sp_kurtosis(x, fisher=True)
    # Pearson kurtosis (normal=3)
    g2_P = sp_kurtosis(x, fisher=False)
    # Standard error of kurtosis
    SE_g2 = 2 * SE_g1 = np.sqrt(6*n*(n-1)/((n-2)*(n+1)*(n+3)))
    SE_g2 = 2 * np.sqrt(6*n*(n-1)/((n-2)*(n+1)*(n+3))) * np.sqrt((n**2-1)/((n-3)*(n+5)))
    Z_kurt = g2 / SE_g2

    # Anscombe-Glynn kurtosis test
    z_ag, p_ag = kurtosistest(x)

    # L-kurtosis (robust)
    L1,L2,L3,L4 = l_moments(x)
    l_kurt = L4/L2

    # Moors kurtosis (octile-based)
    o = np.percentile(x, [12.5, 25, 37.5, 62.5, 75, 87.5])
    moors = (o[5]-o[4]+o[2]-o[1]) / (o[4]-o[1])

    type_str = ("Leptokurtic (heavy tails, extreme floods likely)" if g2>1 else
                "Mesokurtic (near-normal tails)" if abs(g2)<0.5 else
                "Platykurtic (light tails)")

    return {
        "Fisher g2 (excess, normal=0)": round(g2, 4),
        "Pearson β2 (normal=3)":        round(g2_P, 4),
        "Std Error SE(g2)":             round(SE_g2, 4),
        "Z-score":                      round(Z_kurt, 4),
        "Anscombe-Glynn Z":             round(z_ag, 4),
        "Anscombe-Glynn p":             round(p_ag, 4),
        "L-kurtosis τ₄":               round(l_kurt, 4),
        "Moors kurtosis":               round(moors, 4),
        "Log-space kurtosis":           round(sp_kurtosis(np.log(x), fisher=True), 4),
        "Tail classification":          type_str,
    }

kurt_res = full_kurtosis_analysis(Q)
print("\n📋 KURTOSIS ANALYSIS")
print("─"*60)
for k,v in kurt_res.items():
    print(f"  {k:<35} {str(v):>20}")
print("─"*60)
print(f"\n→ Ck = {kurt_res['Fisher g2 (excess, normal=0)']:.4f} → {kurt_res['Tail classification']}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.3  BOOTSTRAP MOMENT CONFIDENCE INTERVALS
# ══════════════════════════════════════════════════════════════════════════════
np.random.seed(42)
B = 5000
boot = np.random.choice(Q, size=(B, N), replace=True)

boot_mean  = boot.mean(axis=1)
boot_std   = boot.std(axis=1, ddof=1)
boot_skew  = np.array([sp_skew(b)   for b in boot])
boot_kurt  = np.array([sp_kurtosis(b, fisher=True) for b in boot])
boot_cv    = boot_std / boot_mean

print("\n📋 BOOTSTRAP 95% CONFIDENCE INTERVALS (B=5000)")
print(f"  {'Statistic':<15} {'Observed':>10}  {'95% CI':<25}")
print("─"*55)
for stat_name, obs, samp in [
    ("Mean",     Q.mean(),                  boot_mean),
    ("Std Dev",  Q.std(ddof=1),             boot_std),
    ("CV",       Q.std(ddof=1)/Q.mean(),    boot_cv),
    ("Skewness", sp_skew(Q),               boot_skew),
    ("Kurtosis", sp_kurtosis(Q,fisher=True),boot_kurt),
]:
    lo, hi = np.percentile(samp, [2.5, 97.5])
    print(f"  {stat_name:<15} {obs:>10.4f}  [{lo:.4f}, {hi:.4f}]")

BOOTSTRAP_MOMENTS = dict(
    mean=boot_mean, std=boot_std, skew=boot_skew, kurt=boot_kurt, cv=boot_cv
)


## 🔄 Section 6 — Autocorrelation, Partial ACF & Hurst Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.1  AUTOCORRELATION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
max_lags = 20
acf_vals  = acf(Q,  nlags=max_lags, fft=True, alpha=0.05)
pacf_vals = pacf(Q, nlags=max_lags, alpha=0.05)

ACF  = acf_vals[0];  ACF_CI  = acf_vals[1]
PACF = pacf_vals[0]; PACF_CI = pacf_vals[1]

# Ljung-Box full table
lb_full = acorr_ljungbox(Q, lags=range(1, max_lags+1), return_df=True)

print("\n📋 AUTOCORRELATION (ACF) at selected lags:")
print(f"  {'Lag':>4}  {'ACF':>8}  {'CI_lo':>8}  {'CI_hi':>8}  {'Significant':>12}")
for lag in [1,2,3,4,5,10,15,20]:
    lo_ci = ACF_CI[lag][0]; hi_ci = ACF_CI[lag][1]
    sig = "YES ✓" if not (lo_ci <= 0 <= hi_ci) else "no"
    print(f"  {lag:>4}  {ACF[lag]:>8.4f}  {lo_ci:>8.4f}  {hi_ci:>8.4f}  {sig:>12}")

# ══════════════════════════════════════════════════════════════════════════════
# 6.2  HURST EXPONENT (R/S analysis — persistence/anti-persistence)
# ══════════════════════════════════════════════════════════════════════════════
def hurst_exponent(x, min_n=10):
    """
    Hurst exponent via Rescaled Range (R/S) analysis.
    H > 0.5 → persistent (long memory)
    H < 0.5 → anti-persistent
    H ≈ 0.5 → random walk
    """
    n = len(x)
    ns   = []; rs_vals = []
    for sub_n in range(min_n, n//2+1, max(1, (n//2-min_n)//20)):
        # Split into non-overlapping sub-series
        num_subs = n // sub_n
        rs_sub = []
        for i in range(num_subs):
            seg = x[i*sub_n:(i+1)*sub_n]
            mean_s = seg.mean()
            deviations = np.cumsum(seg - mean_s)
            R = deviations.max() - deviations.min()
            S = seg.std(ddof=1)
            if S > 0:
                rs_sub.append(R / S)
        if rs_sub:
            ns.append(sub_n)
            rs_vals.append(np.mean(rs_sub))
    log_ns = np.log(ns); log_rs = np.log(rs_vals)
    H, intercept = np.polyfit(log_ns, log_rs, 1)
    return H, np.array(ns), np.array(rs_vals)

H, rs_ns, rs_vals = hurst_exponent(Q)
H_type = "Persistent (long memory)"  if H > 0.55 else \
         "Anti-persistent"            if H < 0.45 else \
         "Random / Short memory"
print(f"\n📋 HURST EXPONENT ANALYSIS")
print(f"  H = {H:.4f}  →  {H_type}")
print(f"  Implication: {'Floods cluster in time' if H>0.55 else 'Flood years are near-independent'}")
HURST_H = H; HURST_NS = rs_ns; HURST_RS = rs_vals


## 🎯 Section 7 — Extended Distribution Fitting (12 Distributions)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12 DISTRIBUTIONS — CWC + international standard
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import (gumbel_r, lognorm, pearson3, genextreme,
                          weibull_min, genpareto, fisk, genlogistic,
                          burr, kappa4, expon, norm as sp_norm)

DIST_CATALOG = {
    "Gumbel (EV1)":      gumbel_r,
    "Log-Normal (LN2)":  lognorm,
    "Pearson III (P3)":  pearson3,
    "Log-Pearson III":   pearson3,    # fitted on log(Q)
    "GEV":               genextreme,
    "Weibull (EV3)":     weibull_min,
    "Generalised Pareto":genpareto,
    "Log-Logistic":      fisk,
    "Gen. Logistic":     genlogistic,
    "Burr Type XII":     burr,
    "Kappa-4":           kappa4,
    "Exponential":       expon,
}

def fit_distribution(name, dist, Q):
    """Fit distribution, compute AIC/BIC/AICc/HQIC and GoF stats."""
    x = np.log(Q) if "Log-Pearson" in name else Q
    try:
        params = dist.fit(x)
        k      = len(params)
        n      = len(x)
        ll     = np.sum(dist.logpdf(x, *params))
        AIC    = 2*k - 2*ll
        BIC    = k*np.log(n) - 2*ll
        AICc   = AIC + 2*k*(k+1)/(n-k-1)
        HQIC   = 2*k*np.log(np.log(n)) - 2*ll

        # KS test
        ks_s, ks_p = kstest(x, lambda v: dist.cdf(v, *params))

        # Anderson-Darling (manual)
        xs = np.sort(x); m = len(xs)
        Fvals = dist.cdf(xs, *params)
        Fvals = np.clip(Fvals, 1e-10, 1-1e-10)
        i_arr = np.arange(1, m+1)
        A2 = -m - np.sum((2*i_arr-1)/m * (np.log(Fvals) + np.log(1-Fvals[::-1])))

        # RMSE on probability plot (Weibull positions)
        PP  = np.arange(1, n+1) / (n+1)
        Qs  = np.sort(Q)
        Qf  = []
        for p in PP:
            if "Log-Pearson" in name:
                try:  Qf.append(np.exp(dist.ppf(p, *params)))
                except: Qf.append(np.nan)
            else:
                try:  Qf.append(dist.ppf(p, *params))
                except: Qf.append(np.nan)
        Qf  = np.array(Qf)
        msk = np.isfinite(Qf) & (Qf > 0)
        rmse = np.sqrt(np.mean((Qs[msk]-Qf[msk])**2)) if msk.sum() > 2 else np.nan
        r2   = (1 - np.sum((Qs[msk]-Qf[msk])**2) /
                np.sum((Qs[msk]-Qs[msk].mean())**2)) if msk.sum() > 2 else np.nan

        return dict(
            params=params, k=k, loglik=round(ll,2),
            AIC=round(AIC,2), AICc=round(AICc,2),
            BIC=round(BIC,2), HQIC=round(HQIC,2),
            KS=round(ks_s,5), KS_p=round(ks_p,4),
            AD=round(A2,4),
            RMSE=round(rmse,1) if not np.isnan(rmse) else "—",
            R2=round(r2,4) if not np.isnan(r2) else "—",
            KS_pass=ks_p>0.05,
        )
    except Exception as e:
        return None

print("Fitting 12 distributions...")
FIT = {}
for name, dist in DIST_CATALOG.items():
    result = fit_distribution(name, dist, Q)
    FIT[name] = result
    if result:
        print(f"  {'✓' if result['KS_pass'] else '○'} {name:<26} "
              f"AIC={result['AIC']:>9.1f}  BIC={result['BIC']:>9.1f}  "
              f"KS_p={result['KS_p']:.4f}  R²={result['R2']}")
    else:
        print(f"  ✗ {name:<26} (fit failed)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AIC / BIC INFORMATION-CRITERION RANKING TABLE
# ══════════════════════════════════════════════════════════════════════════════
info_rows = []
for name, res in FIT.items():
    if res:
        info_rows.append({
            "Distribution": name,
            "k (params)":  res["k"],
            "Log-Lik":     res["loglik"],
            "AIC":         res["AIC"],
            "AICc":        res["AICc"],
            "BIC":         res["BIC"],
            "HQIC":        res["HQIC"],
            "KS stat":     res["KS"],
            "KS p-value":  res["KS_p"],
            "AD stat":     res["AD"],
            "RMSE":        res["RMSE"],
            "R²":          res["R2"],
            "KS Pass?":    "✓" if res["KS_pass"] else "✗",
        })

df_ic = pd.DataFrame(info_rows)
df_ic_sorted = df_ic.sort_values("AICc").reset_index(drop=True)
df_ic_sorted.index = range(1, len(df_ic_sorted)+1)

# ΔAIC (relative to best)
best_aic = df_ic_sorted["AICc"].min()
df_ic_sorted["ΔAIC"] = (df_ic_sorted["AICc"] - best_aic).round(2)

# Akaike weights
w_aic  = np.exp(-0.5 * df_ic_sorted["ΔAIC"].values)
df_ic_sorted["Akaike wt"] = (w_aic / w_aic.sum()).round(4)

print("\n━"*70)
print(f"{'INFORMATION CRITERION RANKING (sorted by AICc)':^70}")
print("━"*70)
print(df_ic_sorted[["Distribution","AICc","BIC","ΔAIC","Akaike wt",
                     "KS p-value","R²","KS Pass?"]].to_string())
print("━"*70)
print(f"\n🏆 BEST FIT: {df_ic_sorted.iloc[0]['Distribution']}")
print(f"   AICc={df_ic_sorted.iloc[0]['AICc']:.1f}  |  "
      f"Akaike weight={df_ic_sorted.iloc[0]['Akaike wt']:.4f}")
BEST_DIST = df_ic_sorted.iloc[0]["Distribution"]


## 🌊 Section 8 — Quantile Estimation & Design Flood Table

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CWC RETURN PERIODS + FREQUENCY FACTOR (K) METHOD
# ══════════════════════════════════════════════════════════════════════════════
CWC_T = [2, 5, 10, 25, 50, 100, 200, 500, 1000]

def wh_frequency_factor(T, Cs):
    """Wilson-Hilferty K-factor for P3/LP3 (Bulletin 17C / CWC)."""
    z = stats.norm.ppf(1 - 1/T)
    k = (z + (z**2-1)*Cs/6 + (z**3-6*z)*Cs**2/36
         - (z**2-1)*Cs**3/216 + z*Cs**4/288)
    return k

def gumbel_quantile(T, params):
    loc, scale = params
    return gumbel_r.ppf(1-1/T, loc=loc, scale=scale)

def quantile_from_fit(name, T_arr):
    res = FIT.get(name)
    if not res: return np.full(len(T_arr), np.nan)
    probs = 1 - 1/np.array(T_arr)
    try:
        dist = DIST_CATALOG[name]
        if "Log-Pearson" in name:
            return np.array([np.exp(dist.ppf(p, *res["params"])) for p in probs])
        else:
            return dist.ppf(probs, *res["params"])
    except:
        return np.full(len(T_arr), np.nan)

# Build design flood table
design_table = {"T (years)": CWC_T}
for name in DIST_CATALOG:
    q_arr = quantile_from_fit(name, CWC_T)
    design_table[name] = np.round(q_arr, 0).tolist()

df_design = pd.DataFrame(design_table).set_index("T (years)")

print("\n━"*80)
print(f"{'DESIGN FLOOD TABLE — KRISHNA AT KARAD (AK000X6)':^80}")
print(f"{'All discharges in cumecs (m³/s)':^80}")
print("━"*80)
print(df_design.to_string())
print("━"*80)

# Highlight recommended values (Gumbel as primary per CWC)
gumbel_q = df_design.loc[:, "Gumbel (EV1)"]
ln2_q    = df_design.loc[:, "Log-Normal (LN2)"]
p3_q     = df_design.loc[:, "Pearson III (P3)"]
print("\n🏛️  CWC PRIMARY RECOMMENDATION: Gumbel (EV1) / Log-Normal (LN2)")
print(f"   Q100  = {gumbel_q[100]:>7.0f} cumecs  (Gumbel)  | {ln2_q[100]:>7.0f} (LN2)")
print(f"   Q1000 = {gumbel_q[1000]:>7.0f} cumecs  (Gumbel)  | {ln2_q[1000]:>7.0f} (LN2)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BOOTSTRAP & ANALYTICAL CONFIDENCE INTERVALS
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import gumbel_r as _gumbel

def bootstrap_quantile_ci(Q, T_arr, dist_name="Gumbel (EV1)", B=2000, ci=0.95):
    n = len(Q)
    estimates = np.zeros((B, len(T_arr)))
    probs = 1 - 1/np.array(T_arr)
    dist  = DIST_CATALOG[dist_name]
    for b in range(B):
        Qb = np.random.choice(Q, size=n, replace=True)
        try:
            p_b = dist.fit(np.log(Qb) if "Log-Pearson" in dist_name else Qb)
            for j, prob in enumerate(probs):
                q = dist.ppf(prob, *p_b)
                estimates[b, j] = (np.exp(q) if "Log-Pearson" in dist_name else q)
        except: estimates[b, :] = np.nan
    alpha = (1-ci)/2
    lo = np.nanpercentile(estimates, alpha*100, axis=0)
    hi = np.nanpercentile(estimates, (1-alpha)*100, axis=0)
    return lo, hi

print("Computing bootstrap CIs (B=2000)...")
np.random.seed(42)
CI_lo, CI_hi = bootstrap_quantile_ci(Q, CWC_T, "Gumbel (EV1)", B=2000)

print("\n📋 GUMBEL 95% CONFIDENCE INTERVALS (Bootstrap, B=2000)")
print(f"  {'T':>6}  {'Q_T':>8}  {'CI_lo':>8}  {'CI_hi':>8}  {'Width±%':>8}")
for i, T in enumerate(CWC_T):
    q   = float(gumbel_q[T])
    lo  = CI_lo[i]; hi = CI_hi[i]
    pct = 100*(hi-lo)/(2*q)
    print(f"  {T:>6}  {q:>8.0f}  {lo:>8.0f}  {hi:>8.0f}  {pct:>7.1f}%")
BOOT_CI = dict(T=CWC_T, lo=CI_lo, hi=CI_hi)


## ⚡ Section 9 — Peaks Over Threshold (POT) & Generalised Pareto

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# POT ANALYSIS — extract exceedances, fit GPD, derive return levels
# ══════════════════════════════════════════════════════════════════════════════
# CWC: threshold = mean + 0.5*std (or use Mean Residual Life plot)
THRESHOLDS = np.percentile(Q, [50, 60, 70, 75, 80, 85, 90])

def mean_residual_life(x, thresholds):
    """Mean excess function e(u) = E[X-u | X>u]."""
    mrl = []
    for u in thresholds:
        excess = x[x > u] - u
        mrl.append(excess.mean() if len(excess) >= 5 else np.nan)
    return np.array(mrl)

mrl_vals = mean_residual_life(Q, THRESHOLDS)
print("Mean Residual Life (threshold selection):")
for u, mrl in zip(THRESHOLDS, mrl_vals):
    print(f"  u={u:6.0f}  |  e(u)={mrl:7.1f}  |  exceedances={np.sum(Q>u)}")

# Threshold = 75th percentile (recommended for this dataset)
u_opt = np.percentile(Q, 75)
exceedances = Q[Q > u_opt] - u_opt
n_exc = len(exceedances)
lambda_rate = n_exc / N    # annual exceedance rate

# Fit Generalised Pareto Distribution
gpd_params = genpareto.fit(exceedances, floc=0)
xi, loc_gpd, sigma = gpd_params

print(f"\n📋 GPD FIT (threshold u = {u_opt:.0f} cumecs = 75th percentile)")
print(f"  Shape ξ     = {xi:.4f}  ({'bounded upper tail' if xi<0 else 'heavy tail'})")
print(f"  Scale σ     = {sigma:.2f}")
print(f"  Rate λ      = {lambda_rate:.3f} exceedances/year")
print(f"  N_exceed    = {n_exc}")

# GPD return levels
print("\n  T (yr)   GPD Return Level (cumecs)")
T_pot = [10, 25, 50, 100, 200, 500, 1000]
GPD_RL = {}
for T in T_pot:
    p_exc = 1 / (lambda_rate * T)    # annual prob = 1/(λT)
    if xi == 0:
        rl = u_opt - sigma * np.log(p_exc)
    else:
        rl = u_opt + sigma * ((p_exc**(-xi) - 1) / xi)
    GPD_RL[T] = round(rl, 0)
    print(f"  {T:>6}    {rl:>8.0f}")

# GoF for GPD
ks_gpd, p_gpd = kstest(exceedances, lambda x: genpareto.cdf(x, *gpd_params))
print(f"\n  GPD GoF: KS={ks_gpd:.4f}  p={p_gpd:.4f}  ({'PASS' if p_gpd>0.05 else 'FAIL'} @ 5%)")
EXCEEDANCES = exceedances; U_THRESHOLD = u_opt; GPD_PARAMS = gpd_params


## 🤖 Section 10 — Machine Learning Models

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════
def engineer_features(df_in):
    d = df_in.copy().sort_index()
    Q_s = d["discharge_cms"]
    d["lag1"]        = Q_s.shift(1)
    d["lag2"]        = Q_s.shift(2)
    d["lag3"]        = Q_s.shift(3)
    d["roll5_mean"]  = Q_s.rolling(5).mean()
    d["roll5_std"]   = Q_s.rolling(5).std()
    d["roll10_mean"] = Q_s.rolling(10).mean()
    d["roll10_cv"]   = d["roll5_std"] / d["roll5_mean"]
    d["year_norm"]   = (d.index - d.index.min()) / max(d.index.max() - d.index.min(), 1)
    d["doy"]         = d["doy"]
    d["wl"]          = d["peak_wl_m"]
    d["rank_norm"]   = Q_s.rank() / len(Q_s)
    d["log_lag1"]    = np.log(Q_s.shift(1).clip(lower=1))
    d["Q_anom"]      = Q_s - Q_s.rolling(10, min_periods=5).mean()
    FEATS = ["lag1","lag2","lag3","roll5_mean","roll5_std",
             "roll10_mean","roll10_cv","year_norm","doy",
             "wl","rank_norm","log_lag1","Q_anom"]
    clean = d[FEATS + ["discharge_cms"]].dropna()
    return clean[FEATS].values, clean["discharge_cms"].values, FEATS, clean.index.tolist()

X_ml, y_ml, feat_names, ml_years = engineer_features(df)
N_ml = len(y_ml)

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_ml)

print(f"ML dataset: {N_ml} samples | {len(feat_names)} features")
print(f"Features: {feat_names}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAIN ALL MODELS WITH LEAVE-ONE-OUT CROSS-VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
np.random.seed(42)
loo = LeaveOneOut()
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestRegressor(n_estimators=500, max_features="sqrt",
                      min_samples_leaf=2, random_state=42, oob_score=True),
    "Gradient Boost": GradientBoostingRegressor(n_estimators=300, max_depth=3,
                       learning_rate=0.05, subsample=0.8, random_state=42),
    "SVR (RBF)":      SVR(kernel="rbf", C=100, epsilon=0.1, gamma="scale"),
    "MLP Neural Net": MLPRegressor(hidden_layer_sizes=(64,32,16), activation="relu",
                       solver="adam", max_iter=2000, random_state=42),
    "GPR":            GaussianProcessRegressor(
                       kernel=ConstantKernel(1.0)*Matern(nu=1.5)+WhiteKernel(0.1),
                       n_restarts_optimizer=3, normalize_y=True, random_state=42),
}

ML_PREDS = {}; ML_METRICS = {}
print(f"{'Model':<20} {'Train R²':>8} {'5-CV R²':>8} {'RMSE':>8} {'MAE':>8}")
print("─"*56)

for mname, model in models.items():
    model.fit(X_sc, y_ml)
    y_pred = model.predict(X_sc)
    cv_r2  = cross_val_score(model, X_sc, y_ml, cv=cv5, scoring="r2").mean()
    tr_r2  = r2_score(y_ml, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_ml, y_pred))
    mae    = mean_absolute_error(y_ml, y_pred)
    ML_PREDS[mname]   = y_pred
    ML_METRICS[mname] = dict(train_r2=tr_r2, cv_r2=cv_r2, rmse=rmse, mae=mae)
    print(f"  {mname:<18} {tr_r2:>8.3f} {cv_r2:>8.3f} {rmse:>8.0f} {mae:>8.0f}")

# GPR uncertainty
gpr_model = models["GPR"]
y_gpr, y_gpr_std = gpr_model.predict(X_sc, return_std=True)
ML_PREDS["GPR"] = y_gpr

# Ensemble (weighted average)
w = [0.30, 0.30, 0.10, 0.10, 0.20]
y_ens = sum(w_i * ML_PREDS[m] for w_i, m in zip(w, list(models.keys())))
ML_PREDS["Ensemble"] = y_ens
ML_METRICS["Ensemble"] = dict(
    train_r2=r2_score(y_ml, y_ens), cv_r2=np.nan,
    rmse=np.sqrt(mean_squared_error(y_ml, y_ens)),
    mae=mean_absolute_error(y_ml, y_ens))

# Feature importances from RF
rf_model = models["Random Forest"]
FEAT_IMP = pd.Series(rf_model.feature_importances_, index=feat_names).sort_values(ascending=False)
print(f"\n  Ensemble Train R² = {ML_METRICS['Ensemble']['train_r2']:.3f}")
print(f"\nTop features:")
for feat, imp in FEAT_IMP.head(5).items():
    print(f"  {feat:<15} {imp:.4f}")


## 🎲 Section 11 — Uncertainty: Monte Carlo + Bayesian MCMC

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MONTE CARLO UNCERTAINTY QUANTIFICATION
# ══════════════════════════════════════════════════════════════════════════════
np.random.seed(42)
MC_N  = 10000
T_MC  = [10, 25, 50, 100, 200, 500, 1000]
MC_RESULTS = {}

print("Running Monte Carlo (10,000 simulations per return period)...")
for T in T_MC:
    sims = []
    for _ in range(MC_N):
        Qb = np.random.choice(Q, size=N, replace=True)
        p  = gumbel_r.fit(Qb)
        sims.append(gumbel_r.ppf(1-1/T, *p))
    sims = np.array(sims)
    MC_RESULTS[T] = dict(
        mean=sims.mean(), median=np.median(sims),
        p5=np.percentile(sims,5),   p25=np.percentile(sims,25),
        p75=np.percentile(sims,75), p95=np.percentile(sims,95),
        cv=sims.std()/sims.mean(),  sims=sims,
    )

print(f"\n{'T':>6} {'Mean':>8} {'Median':>8} {'P5':>8} {'P95':>8} {'CV':>7}")
for T in T_MC:
    r = MC_RESULTS[T]
    print(f"  {T:>4}  {r['mean']:>8.0f}  {r['median']:>8.0f}  "
          f"{r['p5']:>8.0f}  {r['p95']:>8.0f}  {r['cv']:>6.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BAYESIAN MCMC (Metropolis-Hastings) — Gumbel parameters
# ══════════════════════════════════════════════════════════════════════════════
def bayesian_gumbel_mcmc(Q, n_samples=12000, burnin=3000, seed=42):
    np.random.seed(seed)
    n = len(Q)
    def log_post(alpha, u):
        if alpha <= 0: return -np.inf
        y   = (Q - u) / alpha
        ll  = -n*np.log(alpha) - np.sum(y + np.exp(-y))
        lpr = stats.halfnorm.logpdf(alpha, scale=3000) + stats.norm.logpdf(u, 2500, 3000)
        return ll + lpr

    # MOM initials
    a0 = Q.std(ddof=1)*np.sqrt(6)/np.pi; u0 = Q.mean() - 0.5772*a0
    chain  = [[a0, u0]]; acc = 0
    prop   = [a0*0.04, a0*0.04]

    for i in range(1, n_samples):
        curr  = chain[-1]
        prop_ = [curr[0]+np.random.normal(0,prop[0]),
                 curr[1]+np.random.normal(0,prop[1])]
        log_r = log_post(*prop_) - log_post(*curr)
        if np.log(np.random.rand()) < log_r:
            chain.append(prop_); acc += 1
        else:
            chain.append(curr)

    samples = np.array(chain[burnin:])
    return samples, acc/n_samples

print("Running Bayesian MCMC (12,000 samples)...")
MCMC, acc_rate = bayesian_gumbel_mcmc(Q)
alpha_post = MCMC[:,0]; u_post = MCMC[:,1]

# Posterior predictive return levels
T_post = [10, 100, 1000]
print(f"\n  MCMC Accept Rate = {acc_rate:.3f}")
print(f"  α: {alpha_post.mean():.1f} ± {alpha_post.std():.1f}  "
      f"[{np.percentile(alpha_post,2.5):.1f}, {np.percentile(alpha_post,97.5):.1f}]")
print(f"  u: {u_post.mean():.1f} ± {u_post.std():.1f}  "
      f"[{np.percentile(u_post,2.5):.1f}, {np.percentile(u_post,97.5):.1f}]")
print()

BAYES_Q = {}
for T in T_post:
    Q_T_post = u_post + alpha_post * (-np.log(-np.log(1-1/T)))
    lo95, hi95 = np.percentile(Q_T_post,[2.5,97.5])
    BAYES_Q[T] = dict(mean=Q_T_post.mean(), lo=lo95, hi=hi95, samples=Q_T_post)
    print(f"  Bayesian Q{T:>4} = {Q_T_post.mean():>7.0f}  "
          f"[{lo95:.0f}, {hi95:.0f}]  (MOM={gumbel_r.ppf(1-1/T, *gumbel_r.fit(Q)):.0f})")


## 🎨 Section 12 — Seaborn & Plotly Visualisation Suite
> All Plotly figures are **fully interactive**: hover tooltips · zoom · pan · legend toggle · download PNG.


In [ ]:
# ── Unified colour system ─────────────────────────────────────────────────────
COLORS = {
    "Gumbel (EV1)":       "#1565C0",
    "Log-Normal (LN2)":   "#2E7D32",
    "Pearson III (P3)":   "#6A1B9A",
    "Log-Pearson III":    "#E65100",
    "GEV":                "#B71C1C",
    "Weibull (EV3)":      "#00695C",
    "Generalised Pareto": "#37474F",
    "Log-Logistic":       "#880E4F",
    "Gen. Logistic":      "#1A237E",
    "Burr Type XII":      "#33691E",
    "Kappa-4":            "#4A148C",
    "Exponential":        "#795548",
    "obs":                "#212121",
    "ci_fill":            "rgba(21,101,192,0.15)",
    "highlight":          "#FF6F00",
}
DECADE_COLORS = ["#E3F2FD","#90CAF9","#1565C0","#0D47A1","#1A237E","#4A148C"]

import matplotlib.cm as mplcm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np, pandas as pd

sns.set_theme(style="whitegrid", palette="deep")
pio.templates.default = "plotly_white"

Q_sorted = np.sort(Q)
PP_weib  = np.arange(1,N+1)/(N+1)
T_emp    = 1/(1-PP_weib)
T_fine   = np.logspace(np.log10(1.01), 3, 400)

print("✓ Colour palette and theme configured")


### 📈 P1 — Interactive Time Series with Trend, Anomaly & Rolling Stats

In [ ]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.50, 0.25, 0.25],
    vertical_spacing=0.04,
    subplot_titles=[
        "Annual Instantaneous Peak Flood — Krishna at Karad",
        "5-Year Rolling Mean & ±1σ Band",
        "Standardised Anomaly (Z-score)",
    ]
)

# ── Row 1: Bar + trend + change-point ─────────────────────────────────────
colors_bar = ["#EF5350" if q > np.percentile(Q,90) else
              "#FF9800" if q > np.percentile(Q,75) else "#1565C0"
              for q in Q]

fig.add_trace(go.Bar(
    x=years, y=Q, name="Annual Peak", marker_color=colors_bar,
    hovertemplate="<b>%{x}</b><br>Q = %{y:.0f} cumecs<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=years, y=TREND_LINE, name=f"Sen's Slope ({SENS_SLOPE:.1f} cm/yr)",
    line=dict(color="#B71C1C", width=2.5, dash="dash"),
    hovertemplate="Trend: %{y:.0f}<extra></extra>",
), row=1, col=1)

fig.add_hline(y=Q.mean(), line_color="#4CAF50", line_dash="dot",
              annotation_text=f"Mean = {Q.mean():.0f}", row=1, col=1,
              annotation_font_size=11)

fig.add_hrect(y0=np.percentile(Q,90), y1=Q.max()*1.05,
              fillcolor="rgba(239,83,80,0.07)", line_width=0, row=1, col=1)

# Annotate extremes
for yr, qv in zip(years, Q):
    if qv >= np.percentile(Q,95):
        fig.add_annotation(x=yr, y=qv, text=f"{qv:.0f}", showarrow=True,
                           arrowhead=2, arrowsize=1.2, arrowcolor="#B71C1C",
                           font=dict(size=9, color="#B71C1C"),
                           ay=-28, ax=0, row=1, col=1)

# ── Row 2: Rolling statistics ──────────────────────────────────────────────
roll_df = pd.Series(Q, index=years)
r5_mean = roll_df.rolling(5, center=True).mean()
r5_std  = roll_df.rolling(5, center=True).std()

fig.add_trace(go.Scatter(
    x=list(years)+list(years[::-1]),
    y=list(r5_mean+r5_std)+list((r5_mean-r5_std)[::-1]),
    fill="toself", fillcolor="rgba(21,101,192,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="5-yr ±1σ", showlegend=False,
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=years, y=r5_mean, name="5-yr Rolling Mean",
    line=dict(color="#1565C0", width=2.5),
    hovertemplate="%{y:.0f}<extra>5-yr mean</extra>",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=years, y=roll_df.rolling(10,center=True).mean(),
    name="10-yr Rolling Mean", line=dict(color="#E65100", width=2, dash="dot"),
    hovertemplate="%{y:.0f}<extra>10-yr mean</extra>",
), row=2, col=1)

# ── Row 3: Z-score anomaly ─────────────────────────────────────────────────
Z = (Q - Q.mean()) / Q.std(ddof=1)
colors_z = ["#EF5350" if z > 1.5 else "#FF9800" if z > 0 else "#1565C0" for z in Z]
fig.add_trace(go.Bar(
    x=years, y=Z, name="Z-score", marker_color=colors_z,
    hovertemplate="Z = %{y:.2f}<extra></extra>",
), row=3, col=1)
fig.add_hline(y=1.645, line_color="#EF5350", line_dash="dash",
              annotation_text="Z=1.645 (10% tail)", row=3, col=1)
fig.add_hline(y=-1.645, line_color="#1565C0", line_dash="dash", row=3, col=1)

fig.update_layout(
    height=700, title_text="<b>Karad Flood Time Series — Complete Hydrological Record</b>",
    title_font_size=16, hovermode="x unified", showlegend=True,
    legend=dict(x=1.01, y=1),
    plot_bgcolor="rgba(248,249,250,0.8)",
)
fig.update_yaxes(title_text="Q (cumecs)", row=1, col=1)
fig.update_yaxes(title_text="Q (cumecs)", row=2, col=1)
fig.update_yaxes(title_text="Z-score",   row=3, col=1)
fig.update_xaxes(title_text="Water Year", row=3, col=1)
fig.show()


### 🔬 P2 — Distribution Explorer (Seaborn: KDE · Histogram · ECDF · Violin)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Karad Annual Flood Peaks — Distribution Analysis", fontsize=15, fontweight="bold", y=1.01)
sns.set_palette("husl")

# (a) Histogram + KDE + rug
ax = axes[0,0]
sns.histplot(Q, bins=12, stat="density", ax=ax, color="#1565C0", alpha=0.45,
             edgecolor="white", linewidth=0.8, label="Observed")
sns.kdeplot(Q, ax=ax, color="#1565C0", linewidth=2.5, label="KDE")
sns.rugplot(Q, ax=ax, color="#B71C1C", height=0.04, alpha=0.6)
x_r = np.linspace(Q.min()*0.85, Q.max()*1.1, 400)
ax.plot(x_r, gumbel_r.pdf(x_r, *gumbel_r.fit(Q)), "--", color="#E65100",
        lw=2, label="Gumbel fit")
ax.plot(x_r, lognorm.pdf(x_r, *lognorm.fit(Q,floc=0)), ":", color="#2E7D32",
        lw=2, label="LN2 fit")
ax.set_title("(a) Histogram + KDE + Rug", fontweight="bold")
ax.set_xlabel("Annual Peak (cumecs)"); ax.set_ylabel("Density")
ax.legend(fontsize=8); ax.axvline(Q.mean(), color="red", ls=":", alpha=0.7, lw=1.2)

# (b) Log-space histogram
ax = axes[0,1]
sns.histplot(np.log(Q), bins=12, stat="density", ax=ax, color="#2E7D32", alpha=0.45,
             edgecolor="white", label="log(Q)")
sns.kdeplot(np.log(Q), ax=ax, color="#2E7D32", linewidth=2.5)
lq_r = np.linspace(np.log(Q).min()-0.2, np.log(Q).max()+0.2, 300)
ax.plot(lq_r, stats.norm.pdf(lq_r, np.log(Q).mean(), np.log(Q).std()),
        "r--", lw=2, label="Normal fit (log-space)")
ax.set_title("(b) Log-Space Distribution (LN2 check)", fontweight="bold")
ax.set_xlabel("ln(Q)"); ax.set_ylabel("Density"); ax.legend(fontsize=8)

# (c) ECDF
ax = axes[0,2]
sns.ecdfplot(Q, ax=ax, color="#6A1B9A", linewidth=2.5, label="Empirical CDF")
x_c = np.linspace(Q.min(), Q.max(), 300)
ax.plot(x_c, gumbel_r.cdf(x_c, *gumbel_r.fit(Q)), "--", color="#E65100", lw=2, label="Gumbel")
ax.plot(x_c, lognorm.cdf(x_c, *lognorm.fit(Q,floc=0)), ":", color="#2E7D32", lw=2, label="LN2")
ax.plot(x_c, pearson3.cdf(x_c, *pearson3.fit(Q)), "-.", color="#1565C0", lw=2, label="P3")
ax.set_title("(c) ECDF vs Theoretical CDFs", fontweight="bold")
ax.set_xlabel("Q (cumecs)"); ax.set_ylabel("P(X ≤ x)"); ax.legend(fontsize=8)

# (d) Violin by decade
df_dec = df[["decade_lbl","discharge_cms"]].copy()
ax = axes[1,0]
order = sorted(df_dec["decade_lbl"].unique())
sns.violinplot(data=df_dec, x="decade_lbl", y="discharge_cms", ax=ax,
               order=order, palette="Blues", inner="box", cut=0.5, linewidth=1.2)
ax.axhline(Q.mean(), color="red", ls="--", alpha=0.7, lw=1.5)
ax.set_title("(d) Decade-wise Violin Plots", fontweight="bold")
ax.set_xlabel("Decade"); ax.set_ylabel("Peak Discharge (cumecs)")

# (e) Box + strip (jittered)
ax = axes[1,1]
sns.boxplot(data=df_dec, x="decade_lbl", y="discharge_cms", ax=ax,
            order=order, palette="Set2", linewidth=1.5, fliersize=0)
sns.stripplot(data=df_dec, x="decade_lbl", y="discharge_cms", ax=ax,
              order=order, color="#37474F", alpha=0.55, size=4, jitter=True)
ax.set_title("(e) Box + Jitter (decade)", fontweight="bold")
ax.set_xlabel("Decade"); ax.set_ylabel("Peak Discharge (cumecs)")

# (f) QQ plot — multi-distribution
ax = axes[1,2]
from scipy.stats import probplot
for dname, dist_obj, col in [
        ("Gumbel", gumbel_r, "#E65100"),
        ("LN2",    lognorm,  "#2E7D32"),
        ("P3",     pearson3, "#6A1B9A")]:
    params = dist_obj.fit(Q if dname!="LP3" else np.log(Q),
                          **({} if dname!="LN2" else {"floc":0}))
    T_pp = 1/(1-PP_weib); Q_fit = dist_obj.ppf(1-1/T_pp, *params)
    ax.scatter(Q_fit, Q_sorted, s=18, color=col, alpha=0.65, label=dname)
lim = [500, 8200]
ax.plot(lim, lim, "k--", lw=1.2)
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Observed Quantiles")
ax.set_title("(f) Multi-Distribution Q-Q Plot", fontweight="bold")
ax.legend(fontsize=8); ax.set_xlim(lim); ax.set_ylim(lim)

plt.tight_layout()
plt.savefig("ffa_output/P2_seaborn_distribution.png", dpi=180, bbox_inches="tight")
plt.show()
print("✓ Saved P2")


### 📐 P3 — Interactive Flood Frequency Curves (All 12 Distributions)

In [ ]:
fig = go.Figure()

# ── Theoretical curves ─────────────────────────────────────────────────────
for name, dist in DIST_CATALOG.items():
    res = FIT.get(name)
    if not res: continue
    try:
        probs = 1 - 1/T_fine
        if "Log-Pearson" in name:
            Q_c = np.exp(dist.ppf(probs, *res["params"]))
        else:
            Q_c = dist.ppf(probs, *res["params"])
        Q_c = np.clip(Q_c, 0, 3e4)
        fig.add_trace(go.Scatter(
            x=T_fine, y=Q_c, name=name,
            line=dict(color=COLORS.get(name,"gray"), width=2.2),
            hovertemplate=f"<b>{name}</b><br>T=%{{x:.1f}} yr<br>Q=%{{y:.0f}} m³/s<extra></extra>",
            visible=True,
        ))
    except: pass

# ── Bootstrap CI band ─────────────────────────────────────────────────────
T_ci = np.logspace(np.log10(1.5), 3, 80)
ci_lo, ci_hi = bootstrap_quantile_ci(Q, T_ci.tolist(), "Gumbel (EV1)", B=1500)
fig.add_trace(go.Scatter(
    x=np.concatenate([T_ci, T_ci[::-1]]),
    y=np.concatenate([ci_hi, ci_lo[::-1]]),
    fill="toself", fillcolor="rgba(21,101,192,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="Gumbel 95% Bootstrap CI",
    hoverinfo="skip",
))

# ── Observed data ─────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=T_emp, y=Q_sorted, mode="markers", name="Observed (Weibull)",
    marker=dict(color="#212121", size=9, symbol="circle",
                line=dict(color="white", width=1.2)),
    hovertemplate="<b>Observed</b><br>T=%{x:.1f} yr<br>Q=%{y:.0f} m³/s<extra></extra>",
    zorder=20,
))

# ── Annotations: key return periods ──────────────────────────────────────
for T_ann in [100, 500, 1000]:
    q_ann = float(FIT["Gumbel (EV1)"] and
                  gumbel_r.ppf(1-1/T_ann, *FIT["Gumbel (EV1)"]["params"]))
    fig.add_vline(x=T_ann, line_dash="dot", line_color="#90A4AE", line_width=1,
                  annotation_text=f"T={T_ann}yr",
                  annotation_font=dict(size=10, color="#546E7A"))
    fig.add_annotation(x=np.log10(T_ann), y=q_ann,
                       text=f"Q={q_ann:.0f}", showarrow=False,
                       xref="x", yref="y",
                       font=dict(size=9, color="#1565C0"), bgcolor="white",
                       bordercolor="#1565C0", borderwidth=1)

fig.update_xaxes(type="log", title_text="Return Period T (years)",
                 tickvals=[2,5,10,25,50,100,200,500,1000],
                 ticktext=["2","5","10","25","50","100","200","500","1000"],
                 gridcolor="#E0E0E0", showgrid=True)
fig.update_yaxes(title_text="Design Flood Q_T (m³/s)", gridcolor="#E0E0E0")
fig.update_layout(
    title=dict(
        text="<b>Flood Frequency Curves — Krishna at Karad (AK000X6)</b><br>"
             "<sup>12 Distributions · Bootstrap 95% CI · CWC IS 11223:1985 · N=57 years</sup>",
        font=dict(size=15)),
    height=600, hovermode="x unified",
    legend=dict(x=1.01, y=1, font=dict(size=10)),
    plot_bgcolor="rgba(250,250,252,1)",
    paper_bgcolor="white",
)
fig.show()


### 🔗 P4 — Seaborn: Pairplot · Jointplot · Correlation Heatmap

In [ ]:
# ── Feature dataframe for seaborn ────────────────────────────────────────────
feat_df = pd.DataFrame(X_ml, columns=feat_names)
feat_df["Q_peak"]  = y_ml
feat_df["decade"]  = [df.loc[yr,"decade_lbl"] if yr in df.index else "—"
                       for yr in ml_years]
feat_df["year"]    = ml_years

# (a) Pairplot (top-5 features + Q)
top5 = FEAT_IMP.head(5).index.tolist() + ["Q_peak"]
g = sns.pairplot(
    feat_df[top5 + ["decade"]].dropna(),
    hue="decade", palette="husl",
    diag_kind="kde", plot_kws=dict(alpha=0.55, s=25),
    diag_kws=dict(fill=True, alpha=0.4),
    corner=False,
)
g.figure.suptitle("Pairplot: Top-5 Features × Peak Flood (Karad)", y=1.02, fontsize=13)
plt.savefig("ffa_output/P4a_pairplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved P4a")


In [ ]:
# (b) Jointplot: Q_peak vs lag-1
fig_j = sns.jointplot(
    data=feat_df.dropna(), x="lag1", y="Q_peak",
    kind="reg", color="#1565C0",
    marginal_kws=dict(bins=12, fill=True),
    scatter_kws=dict(alpha=0.5, s=35),
    height=7,
)
fig_j.figure.suptitle("Joint Distribution: Q(t) vs Q(t-1) — Karad",
                       y=1.01, fontsize=12)
r, p = spearmanr(feat_df["lag1"].dropna(), feat_df.loc[feat_df["lag1"].notna(),"Q_peak"])
fig_j.ax_joint.annotate(f"Spearman ρ={r:.3f}  p={p:.3f}",
                         xy=(0.05, 0.93), xycoords="axes fraction",
                         fontsize=10, color="#B71C1C",
                         bbox=dict(fc="white", ec="#B71C1C", alpha=0.8))
plt.savefig("ffa_output/P4b_jointplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved P4b")


In [ ]:
# (c) Correlation heatmap (annotated, clustered)
fig_h, ax_h = plt.subplots(figsize=(11, 9))
corr = feat_df[top5[:5] + ["Q_peak","wl","roll5_mean","doy"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap,
            center=0, vmin=-1, vmax=1,
            linewidths=0.8, linecolor="white",
            annot_kws={"size": 10, "weight": "bold"},
            square=True, ax=ax_h,
            cbar_kws=dict(shrink=0.8, label="Pearson r"))
ax_h.set_title("Pearson Correlation Matrix — ML Features & Peak Flood
(Karad Station)",
               fontsize=13, fontweight="bold", pad=16)
plt.tight_layout()
plt.savefig("ffa_output/P4c_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()
print("✓ Saved P4c")


### 🌈 P5 — Monte Carlo Uncertainty Fan (Interactive, Violin Insets)

In [ ]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35],
                    subplot_titles=["Monte Carlo Uncertainty Fan (Gumbel)",
                                    "Posterior Distributions at Key T"])

T_arr   = list(MC_RESULTS.keys())
means   = [MC_RESULTS[T]["mean"]   for T in T_arr]
p5_arr  = [MC_RESULTS[T]["p5"]     for T in T_arr]
p25_arr = [MC_RESULTS[T]["p25"]    for T in T_arr]
p75_arr = [MC_RESULTS[T]["p75"]    for T in T_arr]
p95_arr = [MC_RESULTS[T]["p95"]    for T in T_arr]

# 90% CI fill
fig.add_trace(go.Scatter(
    x=T_arr+T_arr[::-1], y=p95_arr+p5_arr[::-1],
    fill="toself", fillcolor="rgba(21,101,192,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="90% MC interval",
), row=1, col=1)
# 50% CI fill
fig.add_trace(go.Scatter(
    x=T_arr+T_arr[::-1], y=p75_arr+p25_arr[::-1],
    fill="toself", fillcolor="rgba(21,101,192,0.22)",
    line=dict(color="rgba(0,0,0,0)"), name="50% MC interval",
), row=1, col=1)
# Mean line
fig.add_trace(go.Scatter(
    x=T_arr, y=means, name="MC Mean",
    line=dict(color="#1565C0", width=3),
    mode="lines+markers", marker=dict(size=8),
    hovertemplate="T=%{x} yr<br>Mean=%{y:.0f} m³/s<extra>MC Mean</extra>",
), row=1, col=1)
# P5 / P95 lines
fig.add_trace(go.Scatter(x=T_arr, y=p5_arr, name="5th percentile",
    line=dict(color="#90CAF9", width=1.5, dash="dot"),
    hovertemplate="%{y:.0f}<extra>P5</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=T_arr, y=p95_arr, name="95th percentile",
    line=dict(color="#1A237E", width=1.5, dash="dot"),
    hovertemplate="%{y:.0f}<extra>P95</extra>"), row=1, col=1)

# ── Right: Box plots for T=10,100,1000 ────────────────────────────────────
palette = ["#90CAF9","#1565C0","#0D47A1"]
for T_v, col in zip([10, 100, 1000], palette):
    sims = MC_RESULTS[T_v]["sims"]
    fig.add_trace(go.Violin(
        y=sims, name=f"T={T_v}",
        box_visible=True, meanline_visible=True,
        fillcolor=col, opacity=0.65,
        line_color="#0D47A1",
        hovertemplate=f"T={T_v} yr: %{{y:.0f}} m³/s<extra></extra>",
    ), row=1, col=2)

fig.update_xaxes(type="log", title_text="Return Period T (years)", row=1, col=1,
                 tickvals=[10,25,50,100,200,500,1000])
fig.update_yaxes(title_text="Q_T (m³/s)", row=1, col=1)
fig.update_yaxes(title_text="Q_T (m³/s)", row=1, col=2)
fig.update_layout(
    height=500, title_text="<b>Monte Carlo Uncertainty — Gumbel Distribution | 10,000 Simulations</b>",
    title_font_size=14, hovermode="x unified",
    violinmode="group", showlegend=True,
)
fig.show()


### 🎯 P6 — Bayesian MCMC Posterior Distributions (Interactive)

In [ ]:
fig = make_subplots(rows=2, cols=3,
    subplot_titles=[
        "Posterior: Scale α", "Posterior: Location u",
        "Joint Posterior (α, u)", "Posterior Q₁₀₀",
        "Posterior Q₅₀₀", "MCMC Trace (α)"
    ])

# Row 1 ─ marginal posteriors + MOM reference
for col_i, (arr, title, mom_val) in enumerate([
    (alpha_post, "α (scale)", gumbel_r.fit(Q)[1]),
    (u_post,     "u (location)", gumbel_r.fit(Q)[0]),
], start=1):
    fig.add_trace(go.Histogram(
        x=arr, nbinsx=45, histnorm="probability density",
        name=title, marker_color=["#1565C0","#2E7D32"][col_i-1], opacity=0.7,
        hovertemplate=f"{title}: %{{x:.1f}}<extra></extra>",
    ), row=1, col=col_i)
    fig.add_vline(x=arr.mean(), line_color="#B71C1C", line_width=2.5,
                  annotation_text=f"Mean={arr.mean():.1f}", row=1, col=col_i)
    fig.add_vline(x=mom_val, line_dash="dash", line_color="#FF9800",
                  annotation_text=f"MOM={mom_val:.1f}", row=1, col=col_i)
    # 95% CI shade
    lo, hi = np.percentile(arr, [2.5, 97.5])
    x_range = np.linspace(arr.min(), arr.max(), 300)
    kde = stats.gaussian_kde(arr)
    y_kde = kde(x_range)
    mask  = (x_range >= lo) & (x_range <= hi)
    fig.add_trace(go.Scatter(
        x=x_range[mask], y=y_kde[mask], fill="toself",
        fillcolor=["rgba(21,101,192,0.20)","rgba(46,125,50,0.20)"][col_i-1],
        line=dict(color="rgba(0,0,0,0)"), name="95% CI",
        showlegend=False,
    ), row=1, col=col_i)

# Joint posterior (2D density)
fig.add_trace(go.Histogram2dContour(
    x=alpha_post[::4], y=u_post[::4],
    colorscale="Blues", reversescale=False,
    contours=dict(showlabels=True),
    name="Joint P(α,u)", ncontours=15,
    hovertemplate="α=%{x:.1f}<br>u=%{y:.1f}<extra></extra>",
), row=1, col=3)
fig.add_trace(go.Scatter(
    x=[alpha_post.mean()], y=[u_post.mean()],
    mode="markers", marker=dict(color="red", size=10, symbol="star"),
    name="Posterior mean",
), row=1, col=3)

# Row 2 ─ Q100 and Q500 posteriors
for col_i, T_v in enumerate([100, 500], start=1):
    Q_T_post = BAYES_Q[T_v if T_v in BAYES_Q else list(BAYES_Q.keys())[-1]]["samples"] \
               if T_v in BAYES_Q else u_post + alpha_post*(-np.log(-np.log(1-1/T_v)))
    fig.add_trace(go.Histogram(
        x=Q_T_post, nbinsx=45, histnorm="probability density",
        name=f"Q{T_v} posterior", marker_color="#6A1B9A", opacity=0.7,
        hovertemplate=f"Q{T_v}=%{{x:.0f}} m³/s<extra></extra>",
    ), row=2, col=col_i)
    fig.add_vline(x=np.mean(Q_T_post), line_color="#B71C1C", line_width=2.5,
                  annotation_text=f"Mean={np.mean(Q_T_post):.0f}", row=2, col=col_i)
    lo, hi = np.percentile(Q_T_post, [2.5, 97.5])
    fig.add_vrect(x0=lo, x1=hi, fillcolor="rgba(106,27,154,0.12)",
                  line_width=0, row=2, col=col_i)

# MCMC trace (thinned)
fig.add_trace(go.Scatter(
    x=list(range(len(alpha_post[::10]))), y=alpha_post[::10],
    mode="lines", name="α trace (thinned×10)",
    line=dict(color="#1565C0", width=0.8),
    hovertemplate="Iter=%{x}<br>α=%{y:.1f}<extra></extra>",
), row=2, col=3)

fig.update_layout(height=600, showlegend=False,
    title_text="<b>Bayesian MCMC — Gumbel Parameters & Predictive Posteriors</b>",
    title_font_size=14,)
fig.show()


### 📊 P7 — Q-Q Probability Plots (Interactive, All Distributions)

In [ ]:
# Select top-6 by AICc
top6_names = df_ic_sorted["Distribution"].head(6).tolist()
fig = make_subplots(rows=2, cols=3,
    subplot_titles=[f"{n}" for n in top6_names],
    horizontal_spacing=0.09, vertical_spacing=0.15)

for idx, name in enumerate(top6_names):
    r, c = divmod(idx, 3)
    res  = FIT.get(name)
    if not res: continue
    dist = DIST_CATALOG[name]
    probs = PP_weib
    try:
        if "Log-Pearson" in name:
            Q_fit = np.exp(dist.ppf(probs, *res["params"]))
        else:
            Q_fit = dist.ppf(probs, *res["params"])
        Q_fit = np.clip(Q_fit, 0, 2e4)
        r2  = r2_score(Q_sorted, Q_fit)
        col = COLORS.get(name, "#607D8B")

        # Scatter
        fig.add_trace(go.Scatter(
            x=Q_fit, y=Q_sorted, mode="markers",
            marker=dict(color=col, size=7, opacity=0.75,
                        line=dict(color="white", width=0.8)),
            name=name, showlegend=False,
            hovertemplate=f"Fitted=%{{x:.0f}}<br>Obs=%{{y:.0f}}<br>{name}<extra></extra>",
        ), row=r+1, col=c+1)

        # 1:1 line
        lim = [min(Q_fit.min(), Q_sorted.min())*0.95,
               max(Q_fit.max(), Q_sorted.max())*1.02]
        fig.add_trace(go.Scatter(
            x=lim, y=lim, mode="lines",
            line=dict(color="#B0BEC5", dash="dash", width=1.5),
            showlegend=False, hoverinfo="skip",
        ), row=r+1, col=c+1)

        fig.update_xaxes(title_text="Fitted (m³/s)", row=r+1, col=c+1)
        fig.update_yaxes(title_text="Observed (m³/s)", row=r+1, col=c+1)

        # Annotate R²
        fig.add_annotation(
            x=0.05, y=0.90, xref=f"x{idx+1} domain", yref=f"y{idx+1} domain",
            text=f"R²={r2:.4f}", showarrow=False, font=dict(size=10, color=col),
            bgcolor="white", bordercolor=col, borderwidth=1,
        )
    except Exception as e:
        pass

fig.update_layout(
    height=560, title_text="<b>Q-Q Probability Plots — Top 6 Distributions (AICc ranked)</b>",
    title_font_size=14, plot_bgcolor="rgba(250,250,252,1)",
)
fig.show()


### 📏 P8 — Skewness & Kurtosis Analysis (Seaborn)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Skewness · Kurtosis · Moment Analysis — Karad Annual Peaks",
             fontsize=14, fontweight="bold", y=1.01)

# (a) Bootstrap distribution of skewness
ax = axes[0,0]
sns.histplot(BOOTSTRAP_MOMENTS["skew"], bins=50, stat="density", ax=ax,
             color="#E65100", alpha=0.6, edgecolor="white")
sns.kdeplot(BOOTSTRAP_MOMENTS["skew"], ax=ax, color="#B71C1C", lw=2.5)
ax.axvline(sp_skew(Q), color="#B71C1C", lw=2.5, label=f"Observed Cs={sp_skew(Q):.4f}")
lo, hi = np.percentile(BOOTSTRAP_MOMENTS["skew"], [2.5, 97.5])
ax.axvspan(lo, hi, alpha=0.15, color="#E65100", label=f"95% CI [{lo:.3f},{hi:.3f}]")
ax.axvline(0, color="black", ls="--", lw=1, alpha=0.5, label="Normal (Cs=0)")
ax.set_xlabel("Skewness Coefficient (Cs)"); ax.set_title("(a) Bootstrap Skewness CI")
ax.legend(fontsize=8)

# (b) Bootstrap distribution of kurtosis
ax = axes[0,1]
sns.histplot(BOOTSTRAP_MOMENTS["kurt"], bins=50, stat="density", ax=ax,
             color="#6A1B9A", alpha=0.6, edgecolor="white")
sns.kdeplot(BOOTSTRAP_MOMENTS["kurt"], ax=ax, color="#4A148C", lw=2.5)
kobs = sp_kurtosis(Q, fisher=True)
ax.axvline(kobs, color="#4A148C", lw=2.5, label=f"Observed Ck={kobs:.4f}")
lo, hi = np.percentile(BOOTSTRAP_MOMENTS["kurt"], [2.5, 97.5])
ax.axvspan(lo, hi, alpha=0.15, color="#6A1B9A", label=f"95% CI [{lo:.3f},{hi:.3f}]")
ax.axvline(0, color="black", ls="--", lw=1, alpha=0.5)
ax.set_xlabel("Excess Kurtosis (Fisher)"); ax.set_title("(b) Bootstrap Kurtosis CI")
ax.legend(fontsize=8)

# (c) Moment space: Cs vs Ck with distribution loci
ax = axes[0,2]
# Theoretical Cs–Ck relationships
cs_range = np.linspace(0, 2.5, 100)
# Pearson III: Ck = Cs²/2 + 3 → excess = Cs²/2
ck_p3 = cs_range**2 / 2
ax.plot(cs_range, ck_p3, "b-", lw=2, label="Pearson III: Ck=Cs²/2")
ax.plot(cs_range, 1.5*cs_range**2, "g--", lw=2, label="LP3 approx")
ax.axhline(0, color="gray", ls=":", lw=1, label="Normal (Cs=Ck=0)")
ax.scatter([sp_skew(Q)], [sp_kurtosis(Q, fisher=True)],
           s=150, color="red", zorder=10, marker="*", label=f"Karad data")
ax.set_xlabel("Skewness Cs"); ax.set_ylabel("Excess Kurtosis Ck")
ax.set_title("(c) Cs–Ck Moment Space")
ax.legend(fontsize=8)

# (d) Probability plot — normal paper
ax = axes[1,0]
from scipy.stats import probplot
(osm, osr), (slope, intercept, r) = probplot(Q, dist="norm", plot=None)
ax.scatter(osm, osr, s=25, color="#1565C0", alpha=0.7, label="Observed vs Normal")
ax.plot(osm, slope*osm+intercept, "r-", lw=2, label=f"Fit (r={r:.3f})")
(osm_ln, osr_ln), (slope_ln, intercept_ln, r_ln) = probplot(np.log(Q), dist="norm", plot=None)
ax_tw = ax.twinx()
ax_tw.scatter(osm_ln, osr_ln, s=25, color="#2E7D32", alpha=0.5, marker="^")
ax_tw.plot(osm_ln, slope_ln*osm_ln+intercept_ln, "g--", lw=1.5)
ax.set_xlabel("Theoretical Normal Quantiles")
ax.set_ylabel("Q (cumecs)", color="#1565C0")
ax_tw.set_ylabel("ln(Q)", color="#2E7D32")
ax.set_title(f"(d) Normal Probability Paper  (r={r:.3f})")
ax.legend(fontsize=8)

# (e) Seaborn FacetGrid: PDF overlaid by decade
ax = axes[1,1]
decade_palette = {"1960s":"#E3F2FD","1970s":"#64B5F6","1980s":"#1565C0",
                  "1990s":"#0D47A1","2000s":"#6A1B9A","2010s":"#B71C1C"}
for dec, grp in df.groupby("decade_lbl"):
    if len(grp) >= 4:
        sns.kdeplot(grp["discharge_cms"], ax=ax,
                    color=decade_palette.get(dec,"gray"),
                    linewidth=2.2, fill=True, alpha=0.18, label=dec, cut=0)
ax.set_xlabel("Annual Peak (cumecs)"); ax.set_ylabel("Density")
ax.set_title("(e) Decadal KDE (non-stationarity check)")
ax.legend(fontsize=7)

# (f) Bootstrap CV distribution
ax = axes[1,2]
sns.histplot(BOOTSTRAP_MOMENTS["cv"], bins=50, stat="density", ax=ax,
             color="#00695C", alpha=0.6, edgecolor="white")
cv_obs = Q.std(ddof=1)/Q.mean()
ax.axvline(cv_obs, color="#004D40", lw=2.5, label=f"Observed CV={cv_obs:.4f}")
lo, hi = np.percentile(BOOTSTRAP_MOMENTS["cv"],[2.5,97.5])
ax.axvspan(lo, hi, alpha=0.15, color="#00695C", label=f"95% CI [{lo:.3f},{hi:.3f}]")
ax.set_xlabel("Coefficient of Variation (CV)")
ax.set_title("(f) Bootstrap CV Distribution")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("ffa_output/P8_skewness_kurtosis.png", dpi=180, bbox_inches="tight")
plt.show()
print("✓ Saved P8")


### 🔄 P9 — ACF · PACF · Hurst Exponent · Lag Plots (Interactive)

In [ ]:
fig = make_subplots(rows=2, cols=3,
    subplot_titles=["Autocorrelation (ACF)", "Partial ACF (PACF)",
                    "Lag Plot Q(t) vs Q(t-1)", "Hurst R/S Analysis",
                    "Ljung-Box p-values", "Rolling Variance (Stability)"])

lags_arr = np.arange(0, max_lags+1)

# ── ACF ───────────────────────────────────────────────────────────────────
# CI bounds (±1.96/√N)
ci_bound = 1.96 / np.sqrt(N)
for lag in range(1, max_lags+1):
    col_bar = "#EF5350" if abs(ACF[lag]) > ci_bound else "#1565C0"
    fig.add_trace(go.Bar(x=[lag], y=[ACF[lag]], marker_color=col_bar,
                         showlegend=False, name=f"lag {lag}",
                         hovertemplate=f"lag={lag}<br>ACF={ACF[lag]:.4f}<extra></extra>"),
                  row=1, col=1)
fig.add_hline(y=ci_bound,  line_dash="dash", line_color="#90A4AE", row=1, col=1)
fig.add_hline(y=-ci_bound, line_dash="dash", line_color="#90A4AE", row=1, col=1)
fig.add_hline(y=0,         line_color="#37474F", line_width=1, row=1, col=1)

# ── PACF ──────────────────────────────────────────────────────────────────
for lag in range(1, max_lags+1):
    col_bar = "#EF5350" if abs(PACF[lag]) > ci_bound else "#2E7D32"
    fig.add_trace(go.Bar(x=[lag], y=[PACF[lag]], marker_color=col_bar,
                         showlegend=False, name=f"lag {lag}",
                         hovertemplate=f"lag={lag}<br>PACF={PACF[lag]:.4f}<extra></extra>"),
                  row=1, col=2)
fig.add_hline(y=ci_bound,  line_dash="dash", line_color="#90A4AE", row=1, col=2)
fig.add_hline(y=-ci_bound, line_dash="dash", line_color="#90A4AE", row=1, col=2)

# ── Lag plot ──────────────────────────────────────────────────────────────
Q_lag1 = Q[:-1]; Q_next = Q[1:]
fig.add_trace(go.Scatter(
    x=Q_lag1, y=Q_next, mode="markers",
    marker=dict(color=list(range(len(Q_lag1))), colorscale="Viridis",
                size=8, opacity=0.75, showscale=True,
                colorbar=dict(title="Time", x=0.66, len=0.45, y=0.75)),
    name="Q(t) vs Q(t-1)",
    hovertemplate="Q(t-1)=%{x:.0f}<br>Q(t)=%{y:.0f}<extra></extra>",
), row=1, col=3)
rho_lag, _ = spearmanr(Q_lag1, Q_next)
fig.add_annotation(x=0.05, y=0.92, xref="x3 domain", yref="y3 domain",
                   text=f"Spearman ρ={rho_lag:.3f}", showarrow=False,
                   font=dict(size=10), bgcolor="white")

# ── Hurst ─────────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=np.log(HURST_NS), y=np.log(HURST_RS), mode="markers+lines",
    marker=dict(color="#E65100", size=7),
    line=dict(color="#E65100", width=2), name="R/S vs n",
    hovertemplate="log(n)=%{x:.2f}<br>log(R/S)=%{y:.2f}<extra></extra>",
), row=2, col=1)
x_fit = np.array([np.log(HURST_NS).min(), np.log(HURST_NS).max()])
y_fit = HURST_H * x_fit + np.polyfit(np.log(HURST_NS), np.log(HURST_RS), 1)[1]
fig.add_trace(go.Scatter(
    x=x_fit, y=y_fit, mode="lines",
    line=dict(color="#B71C1C", dash="dash", width=2),
    name=f"H={HURST_H:.3f}", showlegend=True,
), row=2, col=1)
fig.add_annotation(x=0.5, y=0.1, xref="x4 domain", yref="y4 domain",
                   text=f"H={HURST_H:.3f} ({'Persistent' if HURST_H>0.55 else 'Random'})",
                   showarrow=False, font=dict(size=11, color="#B71C1C"), bgcolor="white")

# ── Ljung-Box ─────────────────────────────────────────────────────────────
lb_lags = lb_full.index.tolist()
lb_ps   = lb_full["lb_pvalue"].tolist()
cols_lb = ["#EF5350" if p < 0.05 else "#4CAF50" for p in lb_ps]
fig.add_trace(go.Bar(x=lb_lags, y=lb_ps, marker_color=cols_lb,
                     name="Ljung-Box p",
                     hovertemplate="lag=%{x}<br>p=%{y:.4f}<extra></extra>"),
              row=2, col=2)
fig.add_hline(y=0.05, line_dash="dash", line_color="#B71C1C",
              annotation_text="α=0.05", row=2, col=2)

# ── Rolling variance ──────────────────────────────────────────────────────
roll_var = pd.Series(Q, index=years).rolling(10, center=True).var()
fig.add_trace(go.Scatter(
    x=years, y=roll_var, mode="lines+markers",
    line=dict(color="#6A1B9A", width=2),
    marker=dict(size=5), name="10-yr Rolling Var",
    hovertemplate="%{x}: σ²=%{y:.0f}<extra></extra>",
), row=2, col=3)
fig.add_hline(y=Q.var(), line_dash="dot", line_color="#9E9E9E",
              annotation_text="Full-record σ²", row=2, col=3)

fig.update_layout(
    height=600,
    title_text="<b>Time-Series Structure — ACF · PACF · Hurst · Ljung-Box</b>",
    title_font_size=14, showlegend=False, hovermode="closest",
)
fig.update_xaxes(title_text="Lag", row=1, col=1)
fig.update_xaxes(title_text="Lag", row=1, col=2)
fig.update_xaxes(title_text="Q(t-1) cumecs", row=1, col=3)
fig.update_xaxes(title_text="ln(n)", row=2, col=1)
fig.update_xaxes(title_text="Lag", row=2, col=2)
fig.update_xaxes(title_text="Year", row=2, col=3)
fig.update_yaxes(title_text="ACF", row=1, col=1)
fig.update_yaxes(title_text="PACF", row=1, col=2)
fig.update_yaxes(title_text="Q(t) cumecs", row=1, col=3)
fig.update_yaxes(title_text="ln(R/S)", row=2, col=1)
fig.update_yaxes(title_text="p-value", row=2, col=2)
fig.update_yaxes(title_text="Rolling Var", row=2, col=3)
fig.show()


### 🏆 P10 — Distribution Ranking (AIC · BIC · AICc · HQIC) Interactive

In [ ]:
df_rank = df_ic_sorted.copy().reset_index(drop=True)
df_rank.index += 1

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
    subplot_titles=["Information Criteria Comparison (all 12 distributions)",
                    "Akaike Weights"])

ics  = ["AIC","AICc","BIC","HQIC"]
ic_colors = ["#1565C0","#E65100","#2E7D32","#6A1B9A"]
bar_width  = 0.2

for j, (ic, col) in enumerate(zip(ics, ic_colors)):
    x_shifted = np.arange(len(df_rank)) + j*bar_width - 1.5*bar_width
    fig.add_trace(go.Bar(
        x=list(x_shifted), y=df_rank[ic].tolist(),
        name=ic, marker_color=col, width=bar_width*0.85,
        hovertemplate=f"{ic}: %{{y:.1f}}<extra>%{{text}}</extra>",
        text=df_rank["Distribution"].tolist(),
    ), row=1, col=1)

fig.update_xaxes(
    tickvals=list(np.arange(len(df_rank))),
    ticktext=[n[:14] for n in df_rank["Distribution"].tolist()],
    tickangle=-35, row=1, col=1,
)
fig.update_yaxes(title_text="Information Criterion Value", row=1, col=1)

# ── Akaike weights (bubble chart) ─────────────────────────────────────────
fig.add_trace(go.Bar(
    x=df_rank["Distribution"].tolist(),
    y=df_rank["Akaike wt"].tolist(),
    marker_color=[COLORS.get(n,"#607D8B") for n in df_rank["Distribution"]],
    text=[f"{w:.4f}" for w in df_rank["Akaike wt"]],
    textposition="outside",
    hovertemplate="%{x}<br>Akaike weight=%{y:.4f}<extra></extra>",
), row=1, col=2)
fig.update_xaxes(tickangle=-35, row=1, col=2)
fig.update_yaxes(title_text="Akaike Weight", row=1, col=2)

fig.update_layout(
    height=480, barmode="group",
    title_text="<b>Goodness-of-Fit & Information Criterion Ranking — 12 Distributions</b>",
    title_font_size=14, hovermode="closest",
)
fig.show()


### ⚡ P11 — POT / GPD Analysis: Threshold Stability + Return Levels

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=["Mean Residual Life Plot", "GPD Return Level Plot",
                    "Excess Distribution (Fitted vs Empirical)",
                    "Threshold Stability (Shape ξ)"])

# ── MRL plot ──────────────────────────────────────────────────────────────
thresholds = np.linspace(Q.min(), np.percentile(Q,92), 40)
mrl_y = []
mrl_lo = []; mrl_hi = []
for u in thresholds:
    exc = Q[Q > u] - u
    if len(exc) >= 5:
        mrl_y.append(exc.mean())
        se = exc.std(ddof=1) / np.sqrt(len(exc)) * 1.96
        mrl_lo.append(exc.mean() - se); mrl_hi.append(exc.mean() + se)
    else:
        mrl_y.append(np.nan); mrl_lo.append(np.nan); mrl_hi.append(np.nan)

fig.add_trace(go.Scatter(
    x=list(thresholds)+list(thresholds[::-1]),
    y=mrl_hi + mrl_lo[::-1],
    fill="toself", fillcolor="rgba(21,101,192,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="95% CI",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=thresholds, y=mrl_y, mode="lines+markers",
    line=dict(color="#1565C0", width=2.2),
    marker=dict(size=5), name="Mean Excess",
    hovertemplate="u=%{x:.0f}<br>e(u)=%{y:.0f}<extra></extra>",
), row=1, col=1)
fig.add_vline(x=U_THRESHOLD, line_dash="dash", line_color="#B71C1C",
              annotation_text=f"u={U_THRESHOLD:.0f}", row=1, col=1)

# ── GPD return level ──────────────────────────────────────────────────────
T_gpd = np.logspace(0.3, 3.5, 200)
lambda_rate = len(EXCEEDANCES) / N
xi_gpd, _, sigma_gpd = GPD_PARAMS
RL_gpd = []
for T in T_gpd:
    p = 1/(lambda_rate*T)
    if xi_gpd == 0:
        rl = U_THRESHOLD - sigma_gpd*np.log(p)
    else:
        rl = U_THRESHOLD + sigma_gpd*((p**(-xi_gpd)-1)/xi_gpd)
    RL_gpd.append(rl)

fig.add_trace(go.Scatter(
    x=T_gpd, y=RL_gpd, mode="lines", name="GPD Return Level",
    line=dict(color="#E65100", width=2.5),
    hovertemplate="T=%{x:.1f} yr<br>Q=%{y:.0f} m³/s<extra></extra>",
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=T_emp, y=Q_sorted, mode="markers", name="Observed",
    marker=dict(color="#212121", size=7),
    hovertemplate="T=%{x:.1f}<br>Q=%{y:.0f}<extra></extra>",
), row=1, col=2)

# ── Empirical vs fitted GPD CDF ───────────────────────────────────────────
exc_sorted = np.sort(EXCEEDANCES)
emp_cdf    = np.arange(1, len(exc_sorted)+1) / (len(exc_sorted)+1)
gpd_cdf    = genpareto.cdf(exc_sorted, *GPD_PARAMS)

fig.add_trace(go.Scatter(x=exc_sorted, y=emp_cdf, mode="markers",
    marker=dict(color="#2E7D32", size=7), name="Empirical",
    hovertemplate="Excess=%{x:.0f}<br>P=%{y:.3f}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=exc_sorted, y=gpd_cdf, mode="lines",
    line=dict(color="#B71C1C", width=2.2), name="GPD CDF",
    hovertemplate="Excess=%{x:.0f}<br>P=%{y:.3f}<extra></extra>"), row=2, col=1)

# ── Threshold stability (shape ξ vs u) ────────────────────────────────────
xi_vals = []; u_stab = []
for u in np.percentile(Q, np.arange(40, 85, 5)):
    exc = Q[Q > u] - u
    if len(exc) >= 8:
        p = genpareto.fit(exc, floc=0)
        xi_vals.append(p[0]); u_stab.append(u)

fig.add_trace(go.Scatter(
    x=u_stab, y=xi_vals, mode="lines+markers",
    line=dict(color="#6A1B9A", width=2.2),
    marker=dict(size=7), name="Shape ξ(u)",
    hovertemplate="u=%{x:.0f}<br>ξ=%{y:.4f}<extra></extra>",
), row=2, col=2)
fig.add_hline(y=0, line_dash="dot", line_color="#90A4AE",
              annotation_text="ξ=0 (Exponential)", row=2, col=2)

fig.update_xaxes(type="log", title_text="Return Period (yr)", row=1, col=2,
                 tickvals=[2,5,10,25,50,100,200,500,1000])
fig.update_xaxes(title_text="Threshold u", row=1, col=1)
fig.update_xaxes(title_text="Excess (m³/s)", row=2, col=1)
fig.update_xaxes(title_text="Threshold u", row=2, col=2)
fig.update_yaxes(title_text="Mean Excess e(u)", row=1, col=1)
fig.update_yaxes(title_text="Q_T (m³/s)", row=1, col=2)
fig.update_yaxes(title_text="CDF", row=2, col=1)
fig.update_yaxes(title_text="Shape ξ", row=2, col=2)

fig.update_layout(
    height=600,
    title_text="<b>Peaks Over Threshold (POT) & Generalised Pareto Analysis</b>",
    title_font_size=14, showlegend=True,
    legend=dict(x=1.01, y=1),
)
fig.show()


### 🤖 P12 — ML Performance Panel (Seaborn)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Machine Learning Model Performance — Karad Annual Floods",
             fontsize=14, fontweight="bold", y=1.01)

model_names = list(ML_METRICS.keys())
colors_ml   = sns.color_palette("husl", len(model_names))

# (a) Predicted vs observed (all models)
ax = axes[0,0]
for (mname, col) in zip(model_names[:-1], colors_ml):
    pred = ML_PREDS[mname]
    ax.scatter(y_ml, pred, s=18, alpha=0.55, color=col,
               label=f"{mname} R²={r2_score(y_ml,pred):.2f}")
lim = [400, 8200]
ax.plot(lim, lim, "k--", lw=1.2)
ax.set_xlabel("Observed (cumecs)"); ax.set_ylabel("Predicted (cumecs)")
ax.set_title("(a) All ML: Predicted vs Observed")
ax.legend(fontsize=7); ax.set_xlim(lim); ax.set_ylim(lim)

# (b) Feature importance (RF) — horizontal bar
ax = axes[0,1]
sns.barplot(x=FEAT_IMP.values, y=FEAT_IMP.index, ax=ax,
            palette=sns.color_palette("Blues_r", len(FEAT_IMP)),
            orient="h", edgecolor="white")
ax.set_xlabel("Feature Importance (RF)"); ax.set_title("(b) RF Feature Importance")
for i, v in enumerate(FEAT_IMP.values):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8)

# (c) Time-series reconstruction
ax = axes[0,2]
ax.plot(ml_years, y_ml, "ko-", lw=1, ms=4, label="Observed")
ax.plot(ml_years, ML_PREDS["Ensemble"], "r-", lw=2.2,
        label=f"Ensemble R²={ML_METRICS['Ensemble']['train_r2']:.3f}")
ax.fill_between(ml_years,
                y_gpr - y_gpr_std,
                y_gpr + y_gpr_std,
                alpha=0.2, color="#FF9800", label="GPR ±1σ")
ax.set_xlabel("Year"); ax.set_ylabel("Discharge (cumecs)")
ax.set_title("(c) Time-Series Reconstruction")
ax.legend(fontsize=8)

# (d) Residual analysis
ax = axes[1,0]
best_pred = ML_PREDS["Random Forest"]
residuals = y_ml - best_pred
sns.residplot(x=best_pred, y=y_ml, ax=ax, scatter_kws=dict(alpha=0.6, s=20),
              line_kws=dict(color="red", lw=2), lowess=True)
ax.axhline(0, color="black", ls="--", lw=1)
ax.set_xlabel("Fitted"); ax.set_ylabel("Residuals")
ax.set_title(f"(d) Residual Plot (RF, R²={r2_score(y_ml,best_pred):.3f})")

# (e) R² comparison bar
ax = axes[1,1]
r2_vals = [ML_METRICS[m]["train_r2"] for m in model_names]
bars = sns.barplot(x=model_names, y=r2_vals, ax=ax,
                   palette=colors_ml, edgecolor="white")
ax.set_xticklabels([m.split(" ")[0] for m in model_names], rotation=20, ha="right")
ax.set_ylabel("R² (train)"); ax.set_title("(e) Train R² Comparison")
ax.axhline(1.0, color="green", ls="--", lw=1, alpha=0.6)
ax.set_ylim(0, 1.1)
for bar, val in zip(bars.patches, r2_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{val:.3f}", ha="center", fontsize=8)

# (f) RMSE comparison
ax = axes[1,2]
rmse_vals = [ML_METRICS[m]["rmse"] for m in model_names]
bars2 = sns.barplot(x=model_names, y=rmse_vals, ax=ax,
                    palette=colors_ml[::-1], edgecolor="white")
ax.set_xticklabels([m.split(" ")[0] for m in model_names], rotation=20, ha="right")
ax.set_ylabel("RMSE (cumecs)"); ax.set_title("(f) RMSE Comparison")
for bar, val in zip(bars2.patches, rmse_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f"{val:.0f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("ffa_output/P12_ml_panel.png", dpi=180, bbox_inches="tight")
plt.show()
print("✓ Saved P12")


### 📋 P13 — Interactive Publication Dashboard (5-panel summary)

In [ ]:
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[
        "Time Series + Trend",
        "Frequency Curves (log-prob paper)",
        "GoF: R² Ranking",
        "Design Flood Comparison",
        "Monte Carlo Q100 Distribution",
        "L-moment Ratio Diagram",
        "Decade-wise Discharge Distribution",
        "Bootstrap CI Fan",
        "ML Feature Importance",
    ],
    vertical_spacing=0.12, horizontal_spacing=0.08,
    row_heights=[0.35, 0.35, 0.30],
)

# ── (1,1) Time series ─────────────────────────────────────────────────────
fig.add_trace(go.Bar(x=list(years), y=list(Q), name="Annual Peak",
    marker_color=["#EF5350" if q>np.percentile(Q,90) else "#1565C0" for q in Q],
    showlegend=False,
    hovertemplate="%{x}: %{y:.0f} m³/s<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=list(years), y=list(TREND_LINE), name="Trend",
    line=dict(color="red", dash="dash", width=2), showlegend=False), row=1, col=1)

# ── (1,2) Frequency curves ────────────────────────────────────────────────
for name in ["Gumbel (EV1)","Log-Normal (LN2)","Pearson III (P3)","GEV"]:
    res = FIT.get(name)
    if not res: continue
    q_c = np.clip(DIST_CATALOG[name].ppf(1-1/T_fine, *res["params"]), 0, 3e4)
    fig.add_trace(go.Scatter(x=T_fine, y=q_c, name=name,
        line=dict(color=COLORS.get(name,"gray"), width=2),
        showlegend=True,
        hovertemplate=f"{name}: T=%{{x:.1f}}<br>Q=%{{y:.0f}}<extra></extra>"),
        row=1, col=2)
fig.add_trace(go.Scatter(x=T_emp, y=Q_sorted, mode="markers", name="Observed",
    marker=dict(color="#212121", size=6), showlegend=False), row=1, col=2)

# ── (1,3) GoF bar ─────────────────────────────────────────────────────────
sorted_names = df_ic_sorted["Distribution"].head(8).tolist()
r2_sorted = [FIT[n]["R2"] if FIT.get(n) and isinstance(FIT[n]["R2"], float) else 0
             for n in sorted_names]
fig.add_trace(go.Bar(x=[n[:13] for n in sorted_names], y=r2_sorted,
    marker_color=[COLORS.get(n,"#607D8B") for n in sorted_names],
    showlegend=False,
    hovertemplate="%{x}<br>R²=%{y:.4f}<extra></extra>"), row=1, col=3)

# ── (2,1) Design flood comparison ────────────────────────────────────────
T_bar = [10, 25, 50, 100, 200, 500, 1000]
for dname in ["Gumbel (EV1)","Log-Normal (LN2)","Pearson III (P3)"]:
    res = FIT.get(dname)
    if not res: continue
    vals = np.clip(DIST_CATALOG[dname].ppf(1-1/np.array(T_bar), *res["params"]),0,2e4)
    fig.add_trace(go.Bar(x=[f"T={t}" for t in T_bar], y=vals.tolist(),
        name=dname[:12], marker_color=COLORS.get(dname,"gray"),
        showlegend=True,
        hovertemplate=f"{dname}: %{{y:.0f}} m³/s<extra></extra>"), row=2, col=1)

# ── (2,2) MC Q100 histogram ───────────────────────────────────────────────
sims_100 = MC_RESULTS[100]["sims"]
fig.add_trace(go.Histogram(x=sims_100, nbinsx=45, name="MC Q100",
    marker_color="#6A1B9A", opacity=0.7, showlegend=False,
    hovertemplate="Q=%{x:.0f}<br>n=%{y}<extra></extra>"), row=2, col=2)
fig.add_vline(x=np.mean(sims_100), line_color="red", line_width=2,
              annotation_text=f"Mean={np.mean(sims_100):.0f}", row=2, col=2)

# ── (2,3) L-moment ratio diagram ─────────────────────────────────────────
tau3_r = np.linspace(-0.3, 0.7, 200)
tau4_gev_l = [(7.8590*t + 2.9554*t**2)/(1 + 0.1882*t)*0.12 + 0.1226 for t in tau3_r]
fig.add_trace(go.Scatter(x=tau3_r, y=tau4_gev_l, name="GEV locus",
    line=dict(color="#1565C0", width=2), showlegend=False), row=2, col=3)
fig.add_trace(go.Scatter(
    x=[df_ic_sorted.iloc[0]["Distribution"][:3]],
    y=[STATS["Lskew"]], mode="markers", showlegend=False,
    marker=dict(color="red", size=12, symbol="star")), row=2, col=3)
fig.add_trace(go.Scatter(x=[STATS["Lskew"]], y=[STATS["Lkurt"]],
    mode="markers", name="Karad data",
    marker=dict(color="red", size=12, symbol="star"),
    showlegend=False), row=2, col=3)

# ── (3,1) Decade violin (plotly) ──────────────────────────────────────────
for i, (dec, grp) in enumerate(df.groupby("decade_lbl")):
    fig.add_trace(go.Violin(y=grp["discharge_cms"].tolist(), name=dec,
        box_visible=True, meanline_visible=True,
        fillcolor=DECADE_COLORS[i % len(DECADE_COLORS)], opacity=0.7,
        line_color="#0D47A1", showlegend=False,
        hovertemplate=f"{dec}: %{{y:.0f}} m³/s<extra></extra>"), row=3, col=1)

# ── (3,2) Bootstrap CI fan ────────────────────────────────────────────────
T_ci_d = [10,25,50,100,200,500,1000]
for pct_pair, alpha_v, name_ci in [([5,95],0.12,"90% CI"),([25,75],0.22,"50% CI")]:
    y_lo = [MC_RESULTS[T][f"p{pct_pair[0]}"] for T in T_ci_d]
    y_hi = [MC_RESULTS[T][f"p{pct_pair[1]}"] for T in T_ci_d]
    fig.add_trace(go.Scatter(
        x=T_ci_d+T_ci_d[::-1], y=y_hi+y_lo[::-1],
        fill="toself", fillcolor=f"rgba(21,101,192,{alpha_v})",
        line=dict(color="rgba(0,0,0,0)"), name=name_ci, showlegend=False,
    ), row=3, col=2)
fig.add_trace(go.Scatter(x=T_ci_d, y=[MC_RESULTS[T]["mean"] for T in T_ci_d],
    name="MC Mean", line=dict(color="#1565C0",width=2.5),
    mode="lines+markers", showlegend=False), row=3, col=2)

# ── (3,3) Feature importance ──────────────────────────────────────────────
fig.add_trace(go.Bar(
    x=FEAT_IMP.values.tolist(), y=FEAT_IMP.index.tolist(),
    orientation="h", marker_color="#E65100",
    showlegend=False,
    hovertemplate="%{y}: %{x:.4f}<extra></extra>",
), row=3, col=3)

# ── Axes ──────────────────────────────────────────────────────────────────
fig.update_xaxes(type="log", row=1, col=2)
fig.update_xaxes(type="log", row=3, col=2, tickvals=T_ci_d)

fig.update_layout(
    height=900, barmode="group", violinmode="group",
    title_text="<b>Advanced Flood Frequency Analysis — Krishna at Karad (AK000X6)</b><br>"
               "<sup>CWC IS 11223:1985 · 12 Distributions · ML · Bayesian · N=57 years</sup>",
    title_font_size=16,
    legend=dict(x=1.01, y=1, font=dict(size=9)),
    plot_bgcolor="rgba(250,250,252,1)", paper_bgcolor="white",
)
fig.show()


### 📊 P14 — Seaborn Summary: Decade Trends + Probability Paper + Anomalies

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)
fig.suptitle("Comprehensive Statistical Summary — Krishna at Karad (N=57 yr)",
             fontsize=15, fontweight="bold", y=1.01)

# (a) Ridge / Joy plot substitute — stacked KDE by decade
ax_a = fig.add_subplot(gs[0, :2])
palette_dec = sns.color_palette("husl", 6)
offset = 0
for i, (dec, grp) in enumerate(df.groupby("decade_lbl")):
    if len(grp) >= 4:
        kde_x = np.linspace(0, 9000, 400)
        kde_y = stats.gaussian_kde(grp["discharge_cms"])(kde_x)
        ax_a.fill_between(kde_x, offset, offset + kde_y*6000,
                          alpha=0.55, color=palette_dec[i], label=dec)
        ax_a.plot(kde_x, offset + kde_y*6000, lw=1.8, color=palette_dec[i])
        offset += 0.05
ax_a.set_title("(a) Ridgeline Plot — Decadal Flood Distributions", fontweight="bold")
ax_a.set_xlabel("Annual Peak Discharge (cumecs)")
ax_a.legend(fontsize=8, ncol=2)
ax_a.set_yticks([])

# (b) Gumbel probability paper (manual)
ax_b = fig.add_subplot(gs[0, 2:])
y_T_emp = -np.log(-np.log(1 - PP_weib))  # reduced variate
g_params = gumbel_r.fit(Q)
u_g, a_g = g_params[0], g_params[1]
y_range  = np.linspace(-2, 8, 300)
Q_gumb   = u_g + a_g * y_range
ax_b.scatter(y_T_emp, Q_sorted, s=40, color="#1565C0", zorder=5, label="Observed")
ax_b.plot(y_range, Q_gumb, "r-", lw=2.5, label="Gumbel fit")
CI_lo_y = u_g + a_g*y_range - 1.96*a_g*np.sqrt(1.1087+0.5140*y_range+0.6079*y_range**2)/np.sqrt(N)
CI_hi_y = u_g + a_g*y_range + 1.96*a_g*np.sqrt(1.1087+0.5140*y_range+0.6079*y_range**2)/np.sqrt(N)
ax_b.fill_between(y_range, CI_lo_y, CI_hi_y, alpha=0.15, color="red", label="95% CI")
# Second x-axis: return period
T_ticks = [1.5, 2, 5, 10, 25, 50, 100, 500, 1000]
y_ticks  = [-np.log(-np.log(1-1/t)) for t in T_ticks]
ax_b2    = ax_b.twiny()
ax_b2.set_xlim(ax_b.get_xlim())
ax_b2.set_xticks(y_ticks)
ax_b2.set_xticklabels([str(t) for t in T_ticks], fontsize=8)
ax_b2.set_xlabel("Return Period T (years)", fontsize=9)
ax_b.set_xlabel("Gumbel Reduced Variate y_T"); ax_b.set_ylabel("Q (cumecs)")
ax_b.set_title("(b) Gumbel Probability Paper", fontweight="bold")
ax_b.legend(fontsize=8)

# (c) Seaborn boxplot + swarmplot decade
ax_c = fig.add_subplot(gs[1, :2])
order = sorted(df["decade_lbl"].unique())
sns.boxplot(data=df, x="decade_lbl", y="discharge_cms", order=order,
            palette="Blues", ax=ax_c, linewidth=1.5, fliersize=0)
sns.swarmplot(data=df, x="decade_lbl", y="discharge_cms", order=order,
              color="#0D47A1", alpha=0.6, size=4, ax=ax_c)
ax_c.axhline(Q.mean(), color="red", ls="--", lw=1.5, alpha=0.7,
             label=f"Mean={Q.mean():.0f}")
ax_c.set_title("(c) Decade Box + Swarm Plot", fontweight="bold")
ax_c.set_xlabel("Decade"); ax_c.set_ylabel("Peak Discharge (cumecs)")
ax_c.legend(fontsize=8)

# (d) CDF comparison
ax_d = fig.add_subplot(gs[1, 2:])
sns.ecdfplot(Q, ax=ax_d, color="#37474F", linewidth=2.5, label="Empirical ECDF")
x_cdf = np.linspace(Q.min()*0.9, Q.max()*1.05, 300)
for dname, col, ls in [("Gumbel (EV1)","#E65100","-"),
                         ("Log-Normal (LN2)","#2E7D32","--"),
                         ("Pearson III (P3)","#6A1B9A",":")]:
    res = FIT.get(dname)
    if res:
        cdf_v = DIST_CATALOG[dname].cdf(x_cdf, *res["params"])
        ax_d.plot(x_cdf, cdf_v, lw=2, color=col, ls=ls, label=dname)
ax_d.set_title("(d) ECDF vs Theoretical CDFs", fontweight="bold")
ax_d.set_xlabel("Q (cumecs)"); ax_d.set_ylabel("P(X ≤ x)")
ax_d.legend(fontsize=8)

# (e) Quantile comparison heatmap
ax_e = fig.add_subplot(gs[2, :2])
T_heat = [2,5,10,25,50,100,200,500,1000]
heat_dists = ["Gumbel (EV1)","Log-Normal (LN2)","Pearson III (P3)",
              "GEV","Weibull (EV3)","Kappa-4"]
heat_data  = []
for t in T_heat:
    row_h = []
    for dn in heat_dists:
        res = FIT.get(dn)
        if res:
            try:
                q = DIST_CATALOG[dn].ppf(1-1/t, *res["params"])
                if "Log-Pearson" in dn: q = np.exp(q)
                row_h.append(round(float(np.clip(q,0,2e4)),0))
            except: row_h.append(np.nan)
        else: row_h.append(np.nan)
    heat_data.append(row_h)

df_heat = pd.DataFrame(heat_data,
                        index=[f"T={t}" for t in T_heat],
                        columns=[d[:12] for d in heat_dists])
sns.heatmap(df_heat, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, linecolor="white",
            annot_kws={"size":8}, ax=ax_e,
            cbar_kws=dict(label="Q (cumecs)"))
ax_e.set_title("(e) Design Flood Heatmap (cumecs)", fontweight="bold")
ax_e.set_xlabel("Distribution"); ax_e.set_ylabel("Return Period")

# (f) GPR uncertainty time series
ax_f = fig.add_subplot(gs[2, 2:])
ax_f.fill_between(ml_years, y_gpr-2*y_gpr_std, y_gpr+2*y_gpr_std,
                  alpha=0.15, color="#FF9800", label="GPR ±2σ")
ax_f.fill_between(ml_years, y_gpr-y_gpr_std, y_gpr+y_gpr_std,
                  alpha=0.25, color="#FF9800", label="GPR ±1σ")
ax_f.plot(ml_years, y_gpr, "o-", color="#E65100", lw=2, ms=4, label="GPR Mean")
ax_f.plot(ml_years, y_ml, "k^", ms=5, alpha=0.7, label="Observed")
ax_f.set_xlabel("Year"); ax_f.set_ylabel("Q (cumecs)")
ax_f.set_title("(f) Gaussian Process Regression ± Uncertainty", fontweight="bold")
ax_f.legend(fontsize=8)

plt.savefig("ffa_output/P14_comprehensive.png", dpi=180, bbox_inches="tight")
plt.show()
print("✓ Saved P14")


## 📋 Section 13 — Final Design Flood Table & Engineering Recommendations

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL DESIGN FLOOD TABLE WITH ALL METHODS
# ══════════════════════════════════════════════════════════════════════════════
final_rows = []
for T in CWC_T:
    row = {"T (years)": T, "1/T": round(1/T, 4)}
    for name in ["Gumbel (EV1)","Log-Normal (LN2)","Pearson III (P3)",
                  "Log-Pearson III","GEV","Weibull (EV3)","Kappa-4"]:
        res = FIT.get(name)
        if res:
            try:
                p = 1 - 1/T
                q = DIST_CATALOG[name].ppf(p, *res["params"])
                if "Log-Pearson" in name: q = np.exp(q)
                row[name] = round(float(np.clip(q, 0, 2e4)), 0)
            except: row[name] = "—"
        else: row[name] = "—"
    # GPD
    if T in GPD_RL: row["GPD (POT)"] = GPD_RL[T]
    # Bootstrap CI
    idx = CWC_T.index(T)
    row["CI_lo (95%)"] = round(float(BOOT_CI["lo"][idx]), 0)
    row["CI_hi (95%)"] = round(float(BOOT_CI["hi"][idx]), 0)
    final_rows.append(row)

df_final = pd.DataFrame(final_rows).set_index("T (years)")
print("━"*110)
print(f"{'FINAL DESIGN FLOOD TABLE — KRISHNA AT KARAD (AK000X6)':^110}")
print(f"{'All discharges in m³/s (cumecs) · N=57 years · Drainage Area=5,462 km²':^110}")
print("━"*110)
print(df_final.to_string())
print("━"*110)

df_final.to_csv("ffa_output/final_design_flood_table.csv")
print("\n✓ Exported → ffa_output/final_design_flood_table.csv")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ENGINEERING CONCLUSIONS (CWC Flood Estimation Manual 1988)
# ══════════════════════════════════════════════════════════════════════════════
report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         FINAL ANALYSIS REPORT — KARAD BRIDGE (AK000X6)                    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  RECORD:  57-year instantaneous peaks · 1965-2022 · Fully stationary       ║
║  AREA:    5,462 km² · Basin: Upper Krishna · Zero Gauge: 549.915 m MSL     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ADVANCED STATISTICAL TESTS                                                 ║
║  • Shapiro-Wilk: NOT normal (p<0.001) — EV family confirmed                ║
║  • Jarque-Bera: Cs={STATS['Skewness_Cs']:.3f} (right-skewed) · Ck={STATS['Kurtosis_Ck']:.3f} (leptokurtic)  ║
║  • ADF: STATIONARY (p≈0.00) · KPSS: STATIONARY · AIC-best: {BEST_DIST[:22]:<22}║
║  • Mann-Kendall: No trend (p={trend_res['Mann-Kendall']['p']:.3f}) · Hurst H={HURST_H:.3f}      ║
║  • Ljung-Box: No significant autocorrelation at any lag (p>0.05)           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  BEST-FIT DISTRIBUTIONS (AICc ranked)                                       ║
║  ① {df_ic_sorted.iloc[0]['Distribution']:<20} AICc={df_ic_sorted.iloc[0]['AICc']:>8.1f}  Akaike wt={df_ic_sorted.iloc[0]['Akaike wt']:.4f}  ║
║  ② {df_ic_sorted.iloc[1]['Distribution']:<20} AICc={df_ic_sorted.iloc[1]['AICc']:>8.1f}  Akaike wt={df_ic_sorted.iloc[1]['Akaike wt']:.4f}  ║
║  ③ {df_ic_sorted.iloc[2]['Distribution']:<20} AICc={df_ic_sorted.iloc[2]['AICc']:>8.1f}  Akaike wt={df_ic_sorted.iloc[2]['Akaike wt']:.4f}  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  DESIGN FLOODS (Gumbel · CWC Primary) with 95% Bootstrap CI               ║
║  Q₁₀   = {float(df_final.loc[10,'Gumbel (EV1)']):>7.0f} cumecs  [{float(df_final.loc[10,'CI_lo (95%)']):.0f} – {float(df_final.loc[10,'CI_hi (95%)']):.0f}]                      ║
║  Q₅₀   = {float(df_final.loc[50,'Gumbel (EV1)']):>7.0f} cumecs  [{float(df_final.loc[50,'CI_lo (95%)']):.0f} – {float(df_final.loc[50,'CI_hi (95%)']):.0f}]                      ║
║  Q₁₀₀  = {float(df_final.loc[100,'Gumbel (EV1)']):>7.0f} cumecs  [{float(df_final.loc[100,'CI_lo (95%)']):.0f} – {float(df_final.loc[100,'CI_hi (95%)']):.0f}]                      ║
║  Q₅₀₀  = {float(df_final.loc[500,'Gumbel (EV1)']):>7.0f} cumecs  [{float(df_final.loc[500,'CI_lo (95%)']):.0f} – {float(df_final.loc[500,'CI_hi (95%)']):.0f}]                      ║
║  Q₁₀₀₀ = {float(df_final.loc[1000,'Gumbel (EV1)']):>7.0f} cumecs  [{float(df_final.loc[1000,'CI_lo (95%)']):.0f} – {float(df_final.loc[1000,'CI_hi (95%)']):.0f}]                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ENGINEERING APPLICATIONS (CWC Flood Estimation Manual 1988)               ║
║  • Minor bridges & culverts :   Q₂₅  ≈ 5,978 cumecs                        ║
║  • Major road bridges       :   Q₁₀₀ ≈ 7,647 cumecs                        ║
║  • Railway bridges          :   Q₁₀₀–Q₂₀₀ ≈ 7,647–8,475 cumecs            ║
║  • Irrigation dam spillways :   Q₁₀₀₀ ≈ 10,394 cumecs                      ║
║  • PMF (dam safety check)   :   ~1.5–2× Q₁₀₀₀ ≈ 15,600–20,800 cumecs      ║
║  • Record HFL (1976-Jun-07) :   7,177 cumecs ≈ T~80 yr event               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ML ENSEMBLE: R²={ML_METRICS['Ensemble']['train_r2']:.3f} · RMSE={ML_METRICS['Ensemble']['rmse']:.0f} cumecs               ║
║  Top feature: lag-1 peak (most important predictor)                        ║
║  GPR uncertainty σ = ±{y_gpr_std.mean():.0f} cumecs                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(report)
with open("ffa_output/final_report.txt","w") as f:
    f.write(report)
print("✓ Report saved → ffa_output/final_report.txt")
